# Project FLASH — V1: clinical-resolution pneumonia screening, trained for the FPGA

**One sentence:** this notebook reads chest radiographs *as DICOM files*, trains a
47,602-parameter integer CNN whose arithmetic is exactly what the PYNQ-Z2 will compute,
and exports everything the RTL needs to prove it — weights, a layer-descriptor ROM, a
standalone golden model, and bit-exact verification vectors.

Same rule as v0: **this notebook is the single source of truth.** Nothing the RTL
consumes is made by hand.

### What changed from v0, and why

| | v0 | V1 | Why |
|---|---|---|---|
| Data | PneumoniaMNIST, 28×28 | **RSNA Pneumonia DICOMs**, 224×224 (28×28 kept for RTL bring-up) | DICOM is what a hospital PACS actually sends; 28×28 throws away lung texture |
| Input path | pre-made uint8 arrays | real DICOM decode: modality LUT, VOI window, MONOCHROME1, JPEG 2000 — **tested on genuine CR files** | the preprocessing is part of the model; if it is wrong on real scanners, nothing downstream matters |
| Network | conv → pool → flatten → FC16 → FC2, 99.3% of params in FC1 | **5 strided 3×3 convs → global average pool → FC** | flatten+FC hard-codes resolution and dominates quantisation error |
| Requantisation | one fixed `>> 8` | one power-of-two shift **per layer**, calibrated then frozen | layers have different dynamic ranges; shifts cost no multiplier |
| Decision | argmax, no knob | `margin = logit1 − logit0 > T`, **T is a register** | a screening tool must be able to trade specificity for sensitivity |
| Split | official (val ≠ test distribution) | **by patientId**, stratified, val and test from one distribution | v0 §3.3: selecting on the wrong distribution cost 14 points |
| Metrics | balanced accuracy | **AUROC with 95% CI**, sens/spec at a chosen operating point, subgroups (class, AP/PA view, sex) | what a clinical reader asks for |
| Golden check | compared **argmax labels** on 256 images | compares **every intermediate tensor** on the verification set and **logits on the whole val and test sets** | a label can match by luck; a 32-bit logit cannot |
| RTL outputs | padded frames + logits | + per-layer traces, raw accumulators, layer-descriptor ROM | lets a mismatch be localised to one layer, and one stage of that layer |

The last row fixes a small gap in v0: its `torch(hard) vs numpy golden: IDENTICAL` compared
predicted labels, not logits. V1 compares the integers themselves.

### The stage ladder (from the handoff, §4.5)

| `STAGE` | Data | Resolution | Purpose |
|---|---|---|---|
| `V1.0` | PneumoniaMNIST | 28×28 | New architecture on a dataset v0 already trusts. **First RTL target.** |
| `V1.1` | RSNA DICOM | 28×28 | Same network and RTL, new data pipeline. Isolates DICOM bugs from RTL bugs. |
| `V1.2` | RSNA DICOM | 224×224 | The clinical-resolution model. Same RTL engine, bigger counters. |

Every stage writes to its own folder, so running all three does not overwrite anything.
Comparing **V1.1 vs V1.2** is the measured answer to *"does higher resolution help?"* —
same data, same network, only the resolution differs.

### How to run
1. **Runtime → Change runtime type → T4 GPU.**
2. First run: set `SMOKE_TEST = True` in §2. It generates synthetic DICOM files and runs the
   whole notebook in a few minutes with no download. If anything in your environment is
   broken, you find out here, not 40 minutes into training.
3. For RSNA stages: have a Kaggle account, **accept the competition rules** at
   https://www.kaggle.com/competitions/rsna-pneumonia-detection-challenge/rules, and add
   `KAGGLE_USERNAME` / `KAGGLE_KEY` as Colab secrets (🔑 icon in the left bar) — or upload
   `kaggle.json` when prompted.
4. Set `STAGE`, `SMOKE_TEST = False`, then **Runtime → Run all**.

## 1. Setup

Installs only what Colab lacks.

- `pydicom ≥ 3.0` — note that in pydicom 3 the LUT functions moved from
  `pydicom.pixel_data_handlers.util` (the path in the handoff) to **`pydicom.pixels`**. The old
  path is deprecated and will be removed.
- `pylibjpeg-openjpeg`, `pylibjpeg-libjpeg` — decoders for **JPEG 2000** and
  **JPEG-Lossless** transfer syntaxes. RSNA files are uncompressed, but most hospital
  archives are not. Without these, `ds.pixel_array` raises on real CR/DX studies.

In [ ]:
import sys, os, subprocess, importlib, json, math, time, hashlib, shutil, copy, zipfile
from pathlib import Path
IN_COLAB = 'google.colab' in sys.modules

def ensure(module, pip_name):
    try:
        importlib.import_module(module)
    except ImportError:
        print(f'installing {pip_name} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)
        importlib.invalidate_caches()

for module, pip_name in [('pydicom', 'pydicom>=3.0'), ('pylibjpeg', 'pylibjpeg'),
                         ('openjpeg', 'pylibjpeg-openjpeg'), ('libjpeg', 'pylibjpeg-libjpeg'),
                         ('cv2', 'opencv-python-headless'), ('sklearn', 'scikit-learn'),
                         ('pandas', 'pandas'), ('matplotlib', 'matplotlib')]:
    ensure(module, pip_name)

import numpy as np, pandas as pd, cv2, pydicom
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

assert int(pydicom.__version__.split('.')[0]) >= 3, 'need pydicom >= 3 (restart runtime after install)'
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark = True

TOOLS = Path('flash_v1_tools'); TOOLS.mkdir(exist_ok=True)   # files written with %%writefile
sys.path.insert(0, str(TOOLS.resolve()))

print(f'device {DEV} ({torch.cuda.get_device_name(0) if DEV=="cuda" else "no GPU"})')
print(f'torch {torch.__version__} | numpy {np.__version__} | pydicom {pydicom.__version__} | opencv {cv2.__version__}')
if DEV == 'cpu' and IN_COLAB:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU. V1.2 on CPU takes hours.')

## 2. Configuration

The only cell you should need to edit.

| Knob | Meaning |
|---|---|
| `STAGE` | which rung of the ladder (table above) |
| `SMOKE_TEST` | synthetic DICOMs + 4 epochs. Proves the plumbing; its accuracy numbers mean nothing |
| `USE_DRIVE` | cache the *preprocessed* dataset in Google Drive, so the 3.7 GB download and the DICOM decode pass happen once |
| `EPOCHS_SOFT` / `EPOCHS_HARD` | continuous phase, then straight-through integer phase (same idea as v0) |
| `TARGET_SENS` | the sensitivity the default threshold `T` is chosen to reach **on validation** |
| `N_VERIFY` | images written for the Vivado sweep (handoff gate: ≥ 244) |
| `N_TRACE` / `N_ACC` | how many of those also get every intermediate feature map / raw accumulator dumped |

At 224×224, 244 input images are ~37 MB of hex text. That is why traces are limited to a few
images: a trace is for localising a bug, the 244-image sweep is for proving there isn't one.

In [ ]:
STAGE      = 'V1.2'    # 'V1.0' | 'V1.1' | 'V1.2'
SMOKE_TEST = False     # True = synthetic DICOMs, tiny run, no download. Do this once first.
USE_DRIVE  = False     # True = cache preprocessed data in Google Drive

PRESETS = {
    'V1.0': dict(DATASET='pneumoniamnist', IMG=28,  EPOCHS_SOFT=40, EPOCHS_HARD=25, BATCH=128, AUG=False),
    'V1.1': dict(DATASET='rsna',           IMG=28,  EPOCHS_SOFT=30, EPOCHS_HARD=20, BATCH=256, AUG=True),
    'V1.2': dict(DATASET='rsna',           IMG=224, EPOCHS_SOFT=24, EPOCHS_HARD=16, BATCH=128, AUG=True),
}
CFG = dict(STAGE=STAGE, SMOKE_TEST=SMOKE_TEST, **PRESETS[STAGE],
    CHANNELS    = (8, 16, 32, 48, 64),  # conv1..conv5 output channels
    INIT_STD    = 24.0,    # initial weight std in INTEGER units (int8 range is +-127)
    ACT_TARGET  = 128,     # calibration puts the 99th percentile activation here (half of 255 -> headroom)
    LR_SOFT     = 0.30,    # Adam moves each weight ~lr per step; weights are O(10..100)
    LR_HARD     = 0.05,
    TARGET_SENS = 0.90,
    SPLIT       = (0.70, 0.15, 0.15),
    N_VERIFY    = 244, N_TRACE = 8, N_ACC = 2,
    BOOTSTRAP   = 1000,
    EXACT_CHECK_FULL_TEST = True,   # torch(float64) vs golden logits on EVERY test image
    SEED        = 0,
)
if SMOKE_TEST:
    CFG.update(DATASET='rsna', EPOCHS_SOFT=2, EPOCHS_HARD=2, N_VERIFY=16, N_TRACE=2, N_ACC=1,
               BOOTSTRAP=200, SYN_N=400)

np.random.seed(CFG['SEED']); torch.manual_seed(CFG['SEED'])

ROOT   = Path('/content') if IN_COLAB else Path.cwd()
WORK   = ROOT / 'flash_v1'
DATA   = WORK / 'data'
TAG    = STAGE.replace('.', '_').lower() + ('_smoke' if SMOKE_TEST else '')
EXPORT = WORK / 'export' / TAG
for p in [DATA, EXPORT / 'vectors' / 'trace', EXPORT / 'report', EXPORT / 'tools']:
    p.mkdir(parents=True, exist_ok=True)

CACHE = DATA
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path('/content/drive/MyDrive/flash_v1_cache'); CACHE.mkdir(parents=True, exist_ok=True)

IMG = CFG['IMG']
for k, v in CFG.items(): print(f'{k:22s} {v}')
print('export ->', EXPORT)

## 3. The dataset: why RSNA, and what it is not

### Candidates

| Dataset | Format | Labels | Access | Verdict |
|---|---|---|---|---|
| **RSNA Pneumonia Detection** | **DICOM** | radiologists (RSNA + Society of Thoracic Radiology) | Kaggle, accept rules | **chosen** |
| NIH ChestX-ray14 | PNG | mined from reports by NLP (noisy) | open | not DICOM, label noise |
| CheXpert | JPEG | automatic report labeller | registration | not DICOM |
| MIMIC-CXR | DICOM, native | report-derived | PhysioNet credentialing + training course | good later, slow to get |
| VinDr-CXR | DICOM, native hospital output | radiologists | PhysioNet credentialing | **best external test set later** |

RSNA is the right *training* set for a startup today: DICOM, radiologist labels, 26,684
patients, pneumonia is the same task as v0, and one API call downloads it.

### Three things to know before trusting any number from it

1. **The DICOMs are the easy case.** RSNA made them from the NIH PNG release, so they are
   8-bit, MONOCHROME2, already windowed. They exercise the DICOM *container*, not a real
   scanner's pixel encoding. §4 therefore tests the preprocessing against **genuine computed
   radiography files** (15-bit, MONOCHROME1, JPEG 2000) — the ones that break naive loaders.
2. **Prevalence is 22.5%, not 32%.** `stage_2_train_labels.csv` has one row *per bounding
   box*, so pneumonia patients appear 1–4 times. Counting rows gives ~32%; counting patients
   gives ~22.5%. §5 prints both so you can see it.
3. **View position is a shortcut.** AP films are mostly portable bedside films of sicker,
   admitted patients; PA films are mostly walk-in patients. A network can partly learn
   *"this is a portable film"* instead of *"this lung has consolidation"*. §7 measures how much
   AUROC view position alone achieves, and §13 reports AP and PA separately.

### On "clinically viable"
No public-dataset score makes a device clinically viable. What this notebook can give you is
the *evidence a clinical study would start from*: patient-level splits, AUROC with
confidence intervals, an operating point chosen before looking at test, subgroup results, and
a preprocessing path proven on real scanner files. The remaining steps are external validation
on a different hospital's data (VinDr-CXR, or a partner hospital), a reader study, and
regulatory clearance (in India, CDSCO under the Medical Devices Rules, 2017).

## 4. DICOM → model input: the function that ships

The accelerator is bit-exact from the **uint8 tensor** onward. Everything before that tensor
runs on the Zynq's ARM cores — so this function is part of the model, and it is written to a
standalone file (`flash_preprocess.py`) that goes to the board unchanged.

What each step does, in the order DICOM defines (PS3.3 §C.11 / PS3.4 Grayscale Pipeline):

| Step | What | Why it matters |
|---|---|---|
| 1. Decode | `ds.pixel_array` via the transfer syntax | JPEG 2000 / JPEG-Lossless need the pylibjpeg plugins |
| 2. **Modality LUT** | `stored × RescaleSlope + RescaleIntercept` | converts stored integers to meaningful units; must happen *before* windowing |
| 3. **VOI transform** | `WindowCenter`/`WindowWidth`, or a `VOILUTSequence` | maps a 10–16-bit range to what the radiologist sees. Skipping it gives a washed-out image |
| 4. **Photometric** | `MONOCHROME1` → invert | in MONOCHROME1, *low values are white*. Skipping it feeds the network a photographic negative |
| 5. Letterbox | pad to square, centred | real CR is not square (the test file below is 1955×1841). Stretching distorts heart/lung proportions |
| 6. Area resize + round | `cv2.INTER_AREA`, then round to uint8 | area averaging is the correct anti-aliasing filter for large downscales |

If a >8-bit image has no window tags, the function falls back to a 0.5–99.5 percentile
window and **says so** in the returned info, so it is never silent.

**Where exactness ends.** OpenCV's resize is floating point. It is deterministic on one machine,
but an ARM build can differ from x86 by one LSB on a few pixels. The notebook therefore exports
SHA-256 hashes of reference tensors; on the board, preprocess the same files and compare.

In [ ]:
%%writefile flash_v1_tools/flash_preprocess.py
"""Project FLASH V1 -- DICOM -> uint8 model input.

This is the preprocessing contract. The FPGA is bit-exact from the uint8 tensor
onward, so this function is part of the model: the board runs this file unchanged.
"""
import warnings
import numpy as np
import cv2
import pydicom
from pydicom.pixels import apply_modality_lut, apply_voi_lut

# RSNA headers store PatientAge as '51' instead of DICOM's '051Y'; pydicom warns once per file.
warnings.filterwarnings('ignore', message='Invalid value for VR AS')

PREPROC_VERSION = 'flash-v1-preproc-1'


def _first(v):
    """WindowCenter/Width may be multi-valued; DICOM says use the first."""
    try:
        return float(v[0])
    except TypeError:
        return float(v)


def dicom_to_display(ds):
    """Stored pixels -> float64 image in [0, 255], as a radiologist would view it.
    Returns (image, info) where info records which path was taken."""
    arr = ds.pixel_array                                   # 1. decode
    if int(ds.get('NumberOfFrames', 1) or 1) > 1:
        arr = arr[0]
    if int(ds.get('SamplesPerPixel', 1)) != 1:
        raise ValueError('expected a greyscale radiograph (SamplesPerPixel=1)')

    photometric = str(ds.get('PhotometricInterpretation', 'MONOCHROME2')).strip()
    bits_stored = int(ds.get('BitsStored', 8))
    info = dict(photometric=photometric, bits_stored=bits_stored,
                rows=int(arr.shape[0]), cols=int(arr.shape[1]),
                transfer_syntax=str(ds.file_meta.TransferSyntaxUID.name) if 'TransferSyntaxUID' in ds.get('file_meta', {}) else '')

    x = apply_modality_lut(arr, ds).astype(np.float64)    # 2. modality LUT

    if 'VOILUTSequence' in ds and len(ds.VOILUTSequence) > 0:          # 3a. VOI LUT
        lut_bits = int(ds.VOILUTSequence[0].LUTDescriptor[2])
        y = apply_voi_lut(x, ds, index=0, prefer_lut=True).astype(np.float64)
        y = np.clip(y / float((1 << lut_bits) - 1), 0.0, 1.0) * 255.0
        info['voi'] = f'VOILUTSequence({lut_bits} bit)'
    elif 'WindowCenter' in ds and 'WindowWidth' in ds:                  # 3b. window
        c, w = _first(ds.WindowCenter), max(_first(ds.WindowWidth), 1.0)
        fn = str(ds.get('VOILUTFunction', 'LINEAR')).upper()
        if fn == 'SIGMOID':
            y = 255.0 / (1.0 + np.exp(-4.0 * (x - c) / w))
        elif fn == 'LINEAR_EXACT':
            y = np.clip((x - c) / w + 0.5, 0.0, 1.0) * 255.0
        else:   # LINEAR, PS3.3 C.11.2.1.2.1
            y = np.clip((x - (c - 0.5)) / max(w - 1.0, 1e-6) + 0.5, 0.0, 1.0) * 255.0
        info['voi'] = f'window C={c:g} W={w:g} {fn}'
    elif bits_stored <= 8:                                              # 3c. already display-ready
        y = np.clip(x, 0.0, 255.0)
        info['voi'] = 'none (8-bit)'
    else:                                                               # 3d. fallback, flagged
        lo, hi = np.percentile(x, [0.5, 99.5])
        y = np.clip((x - lo) / max(hi - lo, 1e-6), 0.0, 1.0) * 255.0
        info['voi'] = 'PERCENTILE-FALLBACK (no VOI tags)'

    if photometric == 'MONOCHROME1':                                    # 4. invert
        y = 255.0 - y
    return y, info


def letterbox_square(img):
    """5. Pad (not stretch) to a centred square with zeros."""
    h, w = img.shape
    s = max(h, w)
    out = np.zeros((s, s), dtype=img.dtype)
    top, left = (s - h) // 2, (s - w) // 2
    out[top:top + h, left:left + w] = img
    return out, top, left


def to_model_input(display, size):
    """6. Area-resize to size x size and round to uint8."""
    sq, _, _ = letterbox_square(display)
    r = cv2.resize(sq.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA)
    return np.clip(np.rint(r), 0, 255).astype(np.uint8)


def preprocess_dataset(ds, sizes=(224,)):
    """Full pipeline on an opened dataset. Returns ({size: uint8 array}, info)."""
    display, info = dicom_to_display(ds)
    return {s: to_model_input(display, s) for s in sizes}, info


def preprocess_file(path, sizes=(224,)):
    return preprocess_dataset(pydicom.dcmread(path), sizes)


def preprocess_for_training(path, sizes=(28, 224)):
    """Used by the notebook's parallel pass: arrays + the header fields we analyse."""
    ds = pydicom.dcmread(path)
    arrays, info = preprocess_dataset(ds, sizes)
    info.update(patientId=str(ds.get('PatientID', '')), view=str(ds.get('ViewPosition', '')),
                sex=str(ds.get('PatientSex', '')), age=str(ds.get('PatientAge', '')),
                modality=str(ds.get('Modality', '')))
    return arrays, info

In [ ]:
import flash_preprocess as fp
importlib.reload(fp)
print('preprocessing contract:', fp.PREPROC_VERSION)

### 4b. Proving the preprocessing on files that break naive loaders

Each test either passes, fails loudly, or reports **NOT RUN** — never a silent skip.

| Test | File | What it proves |
|---|---|---|
| T1 | real CR, 15-bit, MONOCHROME1, 1955×1841, windowed | the full pipeline on genuine scanner output |
| T2 | the same CR stored as **JPEG 2000 lossless** vs uncompressed | compressed decode is bit-identical |
| T3 | MONOCHROME1 relabelled as MONOCHROME2 | inversion is applied, and only when it should be |
| T4 | RescaleSlope=2, Intercept=−1000, window moved to match | modality LUT runs *before* the window |
| T5 | 8-bit MONOCHROME2, no window (the RSNA case) | the easy case is passed through untouched |
| T6 | non-square 100×60 | letterbox pads instead of stretching |
| T7 | 12-bit with no VOI tags | fallback is used and flagged |

The real CR files come from pydicom's public test corpus (downloaded from GitHub on first use).

In [ ]:
from pydicom.dataset import Dataset, FileMetaDataset
from pydicom.uid import ExplicitVRLittleEndian, generate_uid, SecondaryCaptureImageStorage

def make_dicom(pixels, photometric='MONOCHROME2', bits_stored=None, **tags):
    """Build a minimal valid greyscale DICOM dataset in memory (tests + smoke data)."""
    pixels = np.ascontiguousarray(pixels)
    ds = Dataset()
    ds.file_meta = FileMetaDataset()
    ds.file_meta.TransferSyntaxUID = ExplicitVRLittleEndian
    ds.file_meta.MediaStorageSOPClassUID = SecondaryCaptureImageStorage
    ds.file_meta.MediaStorageSOPInstanceUID = generate_uid()
    ds.SOPClassUID = SecondaryCaptureImageStorage
    ds.SOPInstanceUID = ds.file_meta.MediaStorageSOPInstanceUID
    ds.Modality = tags.pop('Modality', 'CR')
    ds.Rows, ds.Columns = pixels.shape
    ds.SamplesPerPixel = 1
    ds.PhotometricInterpretation = photometric
    ds.BitsAllocated = 8 if pixels.dtype == np.uint8 else 16
    ds.BitsStored = bits_stored or ds.BitsAllocated
    ds.HighBit = ds.BitsStored - 1
    ds.PixelRepresentation = 0
    for k, v in tags.items():
        setattr(ds, k, v)
    ds.PixelData = pixels.tobytes()
    return ds

results = {}
def check(name, ok, detail=''):
    results[name] = 'PASS' if ok else 'FAIL'
    print(f'  {name}: {"PASS" if ok else "FAIL"}  {detail}')

REF_HASHES = {}
try:
    from pydicom.data import get_testdata_file
    rg1_u = pydicom.dcmread(get_testdata_file('RG1_UNCR.dcm'))
    rg1_j = pydicom.dcmread(get_testdata_file('RG1_J2KR.dcm'))
    rg3_u = pydicom.dcmread(get_testdata_file('RG3_UNCR.dcm'))
    rg3_j = pydicom.dcmread(get_testdata_file('RG3_J2KR.dcm'))
    have_real = True
except Exception as e:
    have_real = False
    print('  real CR test files unavailable:', type(e).__name__, e)

if have_real:
    disp1, info1 = fp.dicom_to_display(rg1_u)
    x1 = fp.to_model_input(disp1, IMG)
    print('  RG1:', info1)
    check('T1 real CR end-to-end', x1.shape == (IMG, IMG) and x1.dtype == np.uint8
          and info1['photometric'] == 'MONOCHROME1' and 'window' in info1['voi']
          and 30 < x1.mean() < 225, f'mean={x1.mean():.1f}')

    same_px = all(np.array_equal(a.pixel_array, b.pixel_array) for a, b in [(rg1_u, rg1_j), (rg3_u, rg3_j)])
    same_in = all(np.array_equal(fp.preprocess_dataset(a, (IMG,))[0][IMG], fp.preprocess_dataset(b, (IMG,))[0][IMG])
                  for a, b in [(rg1_u, rg1_j), (rg3_u, rg3_j)])
    check('T2 JPEG 2000 lossless == uncompressed', same_px and same_in,
          f'{rg1_j.file_meta.TransferSyntaxUID.name}')

    m2 = copy.deepcopy(rg1_u); m2.PhotometricInterpretation = 'MONOCHROME2'
    d2, _ = fp.dicom_to_display(m2)
    check('T3 MONOCHROME1 inversion', np.allclose(disp1 + d2, 255.0, atol=1e-9))

    r = copy.deepcopy(rg1_u)
    c, w = fp._first(r.WindowCenter), fp._first(r.WindowWidth)
    r.RescaleSlope, r.RescaleIntercept = 2, -1000
    r.WindowCenter, r.WindowWidth = 2 * c - 1000, 2 * w
    d4, _ = fp.dicom_to_display(r)
    err = np.abs(d4 - disp1).max()
    check('T4 modality LUT before window', err < 0.25, f'max diff {err:.4f} (DICOM +-0.5 offsets)')

    for nm, d in [('RG1_UNCR', rg1_u), ('RG3_UNCR', rg3_u)]:
        for s in (28, 224):
            REF_HASHES[f'{nm}@{s}'] = hashlib.sha256(fp.preprocess_dataset(d, (s,))[0][s].tobytes()).hexdigest()
else:
    for t in ['T1 real CR end-to-end', 'T2 JPEG 2000 lossless == uncompressed',
              'T3 MONOCHROME1 inversion', 'T4 modality LUT before window']:
        results[t] = 'NOT RUN'; print(f'  {t}: NOT RUN')

rng = np.random.default_rng(1)
p8 = rng.integers(0, 256, (64, 64), dtype=np.uint8)
d5, i5 = fp.dicom_to_display(make_dicom(p8))
check('T5 8-bit passthrough (RSNA case)', np.array_equal(d5, p8.astype(np.float64)), i5['voi'])

ns = np.full((100, 60), 200, np.uint8)
x6 = fp.to_model_input(fp.dicom_to_display(make_dicom(ns))[0], 100)
check('T6 letterbox, no stretch', x6[:, :19].max() == 0 and x6[:, 81:].max() == 0 and x6[:, 25:75].min() == 200)

p12 = rng.integers(0, 4096, (64, 64)).astype(np.uint16)
_, i7 = fp.dicom_to_display(make_dicom(p12, bits_stored=12))
check('T7 no-VOI fallback is flagged', 'FALLBACK' in i7['voi'], i7['voi'])

assert 'FAIL' not in results.values(), 'preprocessing self-test failed -- do not train on this pipeline'
print('\npreprocessing self-test:', results)

if have_real:
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(rg1_u.pixel_array, cmap='gray'); ax[0].set_title('stored values, shown naively\n(MONOCHROME1 = a negative)')
    ax[1].imshow(disp1, cmap='gray', vmin=0, vmax=255); ax[1].set_title('after modality LUT + window + invert')
    ax[2].imshow(x1, cmap='gray', vmin=0, vmax=255); ax[2].set_title(f'model input {IMG}x{IMG} uint8 (letterboxed)')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.savefig(EXPORT / 'report' / 'dicom_pipeline.png', dpi=90); plt.show()

## 5. Get the data

Three branches, one output format:

- **`SMOKE_TEST`** writes synthetic radiographs as **real `.dcm` files** plus CSVs with RSNA's
  exact column layout, so everything downstream — DICOM decode, label parsing, splitting —
  runs the same code as the real dataset. The "lungs" are ellipses and "opacities" are bright
  blobs; the images are meant to be learnable, not realistic.
- **RSNA** downloads the competition zip (~3.7 GB) with the Kaggle CLI and extracts only
  the training DICOMs and the two label CSVs. The 3,000 `stage_2_test_images` have no labels
  and are skipped.
- **PneumoniaMNIST** (V1.0 only) is loaded exactly as in v0.

In [ ]:
RSNA_DIR = DATA / ('rsna_synthetic' if SMOKE_TEST else 'rsna')
COMP = 'rsna-pneumonia-detection-challenge'
CLASSES3 = ['Normal', 'No Lung Opacity / Not Normal', 'Lung Opacity']

def synth_cxr(rng, cls, size=512):
    yy, xx = np.mgrid[0:size, 0:size] / size
    img = 150 + 45 * np.exp(-((xx - 0.5) ** 2) / 0.02)                        # mediastinum
    for cx in (0.30, 0.70):                                                    # two lungs
        img -= 95 * np.exp(-(((xx - cx) / 0.14) ** 2 + ((yy - 0.52) / 0.30) ** 2) ** 2)
    noise = cv2.GaussianBlur(rng.normal(0, 1, (size, size)).astype(np.float32), (0, 0), 6)
    img += 60 * noise
    box = None
    if cls == 'Lung Opacity':
        bx, by = rng.choice([0.30, 0.70]) + rng.uniform(-0.05, 0.05), rng.uniform(0.40, 0.65)
        r = rng.uniform(0.05, 0.09)
        img += rng.uniform(45, 75) * np.exp(-((xx - bx) ** 2 + (yy - by) ** 2) / (2 * r * r))
        box = (int((bx - 2 * r) * size), int((by - 2 * r) * size), int(4 * r * size), int(4 * r * size))
    elif cls == 'No Lung Opacity / Not Normal':
        img += 55 * np.exp(-((xx - 0.56) / 0.13) ** 2 - ((yy - 0.66) / 0.10) ** 2)   # "big heart"
    return np.clip(img, 0, 255).astype(np.uint8), box

def build_synthetic(n):
    rng = np.random.default_rng(CFG['SEED'])
    (RSNA_DIR / 'stage_2_train_images').mkdir(parents=True, exist_ok=True)
    lab_rows, cls_rows = [], []
    for i in range(n):
        pid = f'syn{i:05d}-{rng.integers(1 << 30):08x}'
        cls = rng.choice(CLASSES3, p=[0.33, 0.44, 0.23])
        view = 'AP' if rng.random() < (0.80 if cls == 'Lung Opacity' else 0.45) else 'PA'
        px, box = synth_cxr(rng, cls)
        ds = make_dicom(px, PatientID=pid, ViewPosition=view, PatientSex=str(rng.choice(['M', 'F'])),
                        PatientAge=f'{int(rng.integers(18, 90)):03d}Y')
        ds.save_as(RSNA_DIR / 'stage_2_train_images' / f'{pid}.dcm', enforce_file_format=True)
        cls_rows.append((pid, cls))
        if box is None:
            lab_rows.append((pid, np.nan, np.nan, np.nan, np.nan, 0))
        else:
            for _ in range(int(rng.integers(1, 3))):        # 1-2 box rows per positive, like RSNA
                lab_rows.append((pid, *box, 1))
    pd.DataFrame(lab_rows, columns=['patientId', 'x', 'y', 'width', 'height', 'Target']).to_csv(RSNA_DIR / 'stage_2_train_labels.csv', index=False)
    pd.DataFrame(cls_rows, columns=['patientId', 'class']).to_csv(RSNA_DIR / 'stage_2_detailed_class_info.csv', index=False)

def kaggle_setup():
    if (Path.home() / '.kaggle' / 'kaggle.json').exists() or 'KAGGLE_KEY' in os.environ:
        return
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            print('using Kaggle credentials from Colab secrets'); return
        except Exception as e:
            print('no Colab secrets (', type(e).__name__, ') -> upload kaggle.json '
                  '(Kaggle -> Settings -> API -> Create New Token)')
            from google.colab import files
            up = files.upload()
            (Path.home() / '.kaggle').mkdir(exist_ok=True)
            (Path.home() / '.kaggle' / 'kaggle.json').write_bytes(next(iter(up.values())))
            os.chmod(Path.home() / '.kaggle' / 'kaggle.json', 0o600); return
    raise RuntimeError('Kaggle credentials not found (set KAGGLE_USERNAME/KAGGLE_KEY or ~/.kaggle/kaggle.json)')

def download_rsna():
    img_dir = RSNA_DIR / 'stage_2_train_images'
    if img_dir.exists() and len(list(img_dir.glob('*.dcm'))) >= 26684:
        print('RSNA already extracted'); return
    ensure('kaggle', 'kaggle')
    kaggle_setup()
    RSNA_DIR.mkdir(parents=True, exist_ok=True)
    t = time.time()
    r = subprocess.run(['kaggle', 'competitions', 'download', '-c', COMP, '-p', str(RSNA_DIR)],
                       capture_output=True, text=True)
    print(r.stdout[-500:], r.stderr[-500:])
    if r.returncode != 0:
        raise RuntimeError('download failed. If this is a 403: accept the rules at '
                           f'https://www.kaggle.com/competitions/{COMP}/rules and retry.')
    zpath = next(RSNA_DIR.glob('*.zip'))
    print(f'downloaded {zpath.stat().st_size/1e9:.2f} GB in {time.time()-t:.0f}s, extracting ...')
    with zipfile.ZipFile(zpath) as z:
        members = [m for m in z.namelist() if m.startswith('stage_2_train_images/')
                   or m in ('stage_2_train_labels.csv', 'stage_2_detailed_class_info.csv')]
        z.extractall(RSNA_DIR, members=members)
    zpath.unlink()   # free 3.7 GB of Colab disk
    print(f'extracted {len(members)} files in {time.time()-t:.0f}s total')

if CFG['DATASET'] == 'rsna':
    if SMOKE_TEST:
        if not (RSNA_DIR / 'stage_2_detailed_class_info.csv').exists():
            t = time.time(); build_synthetic(CFG['SYN_N'])
            print(f'wrote {CFG["SYN_N"]} synthetic DICOMs in {time.time()-t:.1f}s')
    elif not any(CACHE.glob(f'rsna_{fp.PREPROC_VERSION}.npz')):
        download_rsna()

### 5b. Labels: rows versus patients

Watch the two prevalence lines. This is the 32% vs 22.5% discrepancy from §3, measured.

In [ ]:
if CFG['DATASET'] == 'rsna':
    lab = pd.read_csv(RSNA_DIR / 'stage_2_train_labels.csv')
    cls = pd.read_csv(RSNA_DIR / 'stage_2_detailed_class_info.csv')
    per_patient = lab.groupby('patientId')['Target'].max()
    class3 = cls.drop_duplicates('patientId').set_index('patientId')['class']
    assert cls.groupby('patientId')['class'].nunique().max() == 1, 'a patient has conflicting classes'
    assert ((class3.loc[per_patient.index] == 'Lung Opacity') == (per_patient == 1)).all(), \
        'Target and class disagree'
    print(f'label rows            : {len(lab):6d}   positive rows    : {lab.Target.mean():.1%}   <- inflated by multi-box rows')
    print(f'unique patients       : {per_patient.size:6d}   positive patients: {per_patient.mean():.1%}   <- the real prevalence')
    print('\nthree-way class, per patient:')
    print(class3.value_counts().reindex(CLASSES3).to_string())

## 6. Decode every DICOM once, cache the tensors

Every DICOM goes through `fp.preprocess_for_training` — **the same function tested in §4** —
producing both the 28×28 and 224×224 tensors in one pass, so V1.1 and V1.2 see identical
source images. The 28×28 version is resized from the full-resolution image, not from 224.

The header census below is the evidence for §3's claim that RSNA is the easy case: expect one
row each for photometric interpretation, bits stored and transfer syntax.

This takes a few minutes on Colab's 2 CPUs the first time; after that the `.npz` cache loads in
seconds (and survives runtime resets if `USE_DRIVE = True`).

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

if CFG['DATASET'] == 'rsna':
    cache_file = CACHE / f'rsna_{"synthetic_" if SMOKE_TEST else ""}{fp.PREPROC_VERSION}.npz'
    if cache_file.exists():
        z = np.load(cache_file, allow_pickle=False)
        X28, X224 = z['X28'], z['X224']
        meta = pd.read_csv(cache_file.with_suffix('.csv'), dtype={'age': str})
        print('loaded cache', cache_file)
    else:
        paths = sorted((RSNA_DIR / 'stage_2_train_images').glob('*.dcm'))
        t = time.time()
        n_workers = max(1, os.cpu_count() or 1)
        X28 = np.zeros((len(paths), 1, 28, 28), np.uint8)
        X224 = np.zeros((len(paths), 1, 224, 224), np.uint8)
        infos = []
        with ProcessPoolExecutor(n_workers, mp_context=mp.get_context('spawn')) as ex:
            # the worker must be an importable module function: spawned processes cannot see notebook globals
            for i, (arrays, info) in enumerate(ex.map(fp.preprocess_for_training, map(str, paths), chunksize=32)):
                X28[i, 0], X224[i, 0] = arrays[28], arrays[224]
                info['file_stem'] = paths[i].stem
                infos.append(info)
                if (i + 1) % 5000 == 0:
                    print(f'  {i+1}/{len(paths)}  {time.time()-t:.0f}s')
        meta = pd.DataFrame(infos)
        np.savez(cache_file, X28=X28, X224=X224)
        meta.to_csv(cache_file.with_suffix('.csv'), index=False)
        print(f'decoded {len(paths)} DICOMs with {n_workers} workers in {time.time()-t:.0f}s -> {cache_file}')

    # labels are keyed by filename; the DICOM header's PatientID should agree with it
    mism = int((meta['patientId'].astype(str) != meta['file_stem'].astype(str)).sum())
    print(f'header PatientID differs from filename in {mism} of {len(meta)} files')
    meta['patientId'] = meta['file_stem'].astype(str)
    meta['class3'] = class3.reindex(meta['patientId']).values
    meta['target'] = (meta['class3'] == 'Lung Opacity').astype(np.int64)
    assert meta['class3'].notna().all(), 'image without a label'
    assert meta['patientId'].is_unique, 'more than one image per patient -- split logic must change'

    print('\nDICOM header census (one row per value = homogeneous dataset):')
    for col in ['modality', 'photometric', 'bits_stored', 'transfer_syntax', 'voi', 'view', 'sex']:
        print(f'  {col:16s}', meta[col].value_counts().to_dict())
    ages = pd.to_numeric(meta['age'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
    print(f'  age              median {ages.median():.0f}, range {ages.min():.0f}-{ages.max():.0f}')
    X_ALL = X224 if IMG == 224 else X28
    Y_ALL = meta['target'].values

In [ ]:
if CFG['DATASET'] == 'pneumoniamnist':
    ensure('medmnist', 'medmnist==3.0.2')
    from medmnist import PneumoniaMNIST
    def pack(ds):
        return (np.asarray(ds.imgs, np.uint8).reshape(-1, 1, 28, 28), np.asarray(ds.labels, np.int64).reshape(-1))
    parts = {s: pack(PneumoniaMNIST(split=s, download=True)) for s in ['train', 'val', 'test']}
    X_ALL = np.concatenate([parts[s][0] for s in parts]); Y_ALL = np.concatenate([parts[s][1] for s in parts])
    n_tr, n_va = len(parts['train'][0]), len(parts['val'][0])
    meta = pd.DataFrame(dict(patientId=[f'pmnist{i}' for i in range(len(Y_ALL))], target=Y_ALL,
                             class3=np.where(Y_ALL == 1, 'Lung Opacity', 'Normal'), view='', sex=''))
    print({s: parts[s][0].shape for s in parts})

### 6b. See the resolution difference

The whole premise of V1.2 in one figure. The red box is the radiologist's opacity annotation,
scaled into model-input coordinates. At 28×28 each pixel covers ~37×37 original pixels.

In [ ]:
if CFG['DATASET'] == 'rsna':
    rng = np.random.default_rng(3)
    fig, ax = plt.subplots(3, 4, figsize=(12, 9.5))
    for r, c3 in enumerate(CLASSES3):
        picks = rng.choice(np.where(meta['class3'] == c3)[0], 2, replace=False)
        for j, i in enumerate(picks):
            for k, (arr, s) in enumerate([(X28, 28), (X224, 224)]):
                a = ax[r, 2 * j + k]
                a.imshow(arr[i, 0], cmap='gray', vmin=0, vmax=255, interpolation='nearest')
                a.set_title(f'{c3[:22]} | {s}x{s} | {meta.view[i]}', fontsize=8); a.axis('off')
                if c3 == 'Lung Opacity':
                    R, C = meta.rows[i], meta.cols[i]; S = max(R, C); sc = s / S
                    oy, ox = (S - R) / 2, (S - C) / 2
                    for _, b in lab[(lab.patientId == meta.patientId[i]) & (lab.Target == 1)].iterrows():
                        a.add_patch(plt.Rectangle(((b.x + ox) * sc - 0.5, (b.y + oy) * sc - 0.5), b.width * sc, b.height * sc,
                                                  fill=False, ec='red', lw=1))
    plt.tight_layout(); plt.savefig(EXPORT / 'report' / 'samples_28_vs_224.png', dpi=90); plt.show()

## 7. Split by patient — and measure the shortcut

**Split.** 70/15/15, stratified on the *three-way* class so the hard negatives
(*No Lung Opacity / Not Normal*) are proportionally represented in every split. RSNA has exactly
one image per patient (asserted in §6), so a patient split and an image split coincide here —
but the code asserts disjointness by `patientId`, so it stays correct if a dataset with
repeat studies is ever swapped in.

**Why val and test must share a distribution** (handoff §3.3): the epoch and the threshold `T`
are both chosen on validation. If validation differs from test, both choices are tuned to the
wrong target. Here both are random draws from the same pool, so what you select is what you get.

**The shortcut number.** `AUROC(view position alone)` treats "is this an AP film?" as a
pneumonia score. If it is well above 0.5, part of any model's AUROC can come from recognising
the film type rather than the lung. It is a floor to beat, and the reason §13 reports AP and PA
separately.

In [ ]:
SEED = CFG['SEED']
if CFG['DATASET'] == 'rsna':
    idx = np.arange(len(meta)); strat = meta['class3'].values
    tr_idx, tmp = train_test_split(idx, test_size=CFG['SPLIT'][1] + CFG['SPLIT'][2], stratify=strat, random_state=SEED)
    va_idx, te_idx = train_test_split(tmp, test_size=CFG['SPLIT'][2] / (CFG['SPLIT'][1] + CFG['SPLIT'][2]),
                                      stratify=strat[tmp], random_state=SEED)
else:
    tr_idx = np.arange(n_tr); va_idx = np.arange(n_tr, n_tr + n_va); te_idx = np.arange(n_tr + n_va, len(Y_ALL))
    print('PneumoniaMNIST: official splits (no patient IDs). Val and test come from different '
          'collections (handoff 3.3) -- treat the operating point on test with suspicion.')

P = meta['patientId'].values
assert not (set(P[tr_idx]) & set(P[va_idx]) or set(P[tr_idx]) & set(P[te_idx]) or set(P[va_idx]) & set(P[te_idx])), 'patient leak'

Xtr, Ytr = X_ALL[tr_idx], Y_ALL[tr_idx]
Xva, Yva = X_ALL[va_idx], Y_ALL[va_idx]
Xte, Yte = X_ALL[te_idx], Y_ALL[te_idx]
mva, mte = meta.iloc[va_idx].reset_index(drop=True), meta.iloc[te_idx].reset_index(drop=True)

rows = []
for nm, ii in [('train', tr_idx), ('val', va_idx), ('test', te_idx)]:
    m = meta.iloc[ii]
    row = dict(split=nm, patients=len(ii), positive=int(m.target.sum()), prevalence=f'{m.target.mean():.1%}')
    for c3 in CLASSES3: row[c3[:14]] = int((m.class3 == c3).sum())
    if CFG['DATASET'] == 'rsna': row['AP share'] = f'{(m.view == "AP").mean():.1%}'
    rows.append(row)
print(pd.DataFrame(rows).to_string(index=False))

if CFG['DATASET'] == 'rsna':
    ap = (meta['view'] == 'AP').astype(int).values
    SHORTCUT_AUROC = roc_auc_score(Y_ALL[te_idx], ap[te_idx])
    print(f'\nAUROC of "view == AP" alone, on test: {SHORTCUT_AUROC:.3f}   (0.5 = no information)')
    print(f'P(opacity | AP) = {Y_ALL[ap==1].mean():.1%}    P(opacity | PA) = {Y_ALL[ap==0].mean():.1%}')
else:
    SHORTCUT_AUROC = None

## 8. The network, and why it looks like this

| Layer | Op | Output @224 | Output @28 | Params |
|---|---|---|---|---|
| conv1 | 3×3 stride 2, 1→8 | 8×112×112 | 8×14×14 | 72 + 8 |
| conv2 | 3×3 stride 2, 8→16 | 16×56×56 | 16×7×7 | 1,152 + 16 |
| conv3 | 3×3 stride 2, 16→32 | 32×28×28 | 32×4×4 | 4,608 + 32 |
| conv4 | 3×3 stride 2, 32→48 | 48×14×14 | 48×2×2 | 13,824 + 48 |
| conv5 | 3×3 stride 2, 48→64 | 64×7×7 | 64×1×1 | 27,648 + 64 |
| GAP | sum, `>> s_gap` | 64 | 64 | 0 |
| fc | 64→2 | 2 logits | 2 logits | 128 + 2 |

**Why global average pooling changes what the hardware has to do.** In v0, flatten+FC1 was
12,544 of 12,634 parameters — a 784-input neuron whose weight count *is* the image size. The
hardware had to store the whole pooled feature map and stream 784 weights per neuron. With GAP
the classifier sees 64 numbers regardless of resolution: the hardware keeps 64 running sums, and
the FC layer is 128 weights. The same weights file is legal at 28, 224 or 512; only the loop
counters in the layer table change.

*Honest caveat:* "resolution-agnostic" is true of the **bitstream and the weight shapes**, not of
accuracy. A model trained at 224 has learned features at 224's scale, and `s_gap` changes with
the final map size. Moving to 512 means re-training (or at minimum re-validating) — same RTL,
new weights file.

**Why strided convolutions instead of conv + max-pool.** A stride-2 conv computes 1/4 of the
windows a stride-1 conv would — at 224 that is 12.2 M multiply-accumulates instead of ~49 M —
and it removes the pooling block that caused v0 defect #2. One operator type means one engine.

**Why these channel counts.** 47.6 k INT8 parameters ≈ 47 KB ≈ 11 BRAM36 tiles, so every weight
lives on-chip permanently and never streams from DDR (handoff §4.2). Doubling channels
roughly quadruples parameters and would start to crowd the feature-map buffers.

**28×28 caveat.** Five stride-2 layers take 28 → 14 → 7 → 4 → 2 → 1. conv5 sees a 2×2 map that is
mostly padding, and GAP averages one pixel. That is fine for V1.0, whose job is to bring up the
*same engine* in RTL; do not read V1.0's accuracy as this architecture's accuracy. V1.0 also
exercises something V1.2 never does: **conv3's input is 7×7, an odd size**, so its last window
uses the bottom/right padding row. At 224 every conv input is even and that padding is never
touched. The RTL must handle both.

### The integer arithmetic contract

This is exactly what training's hard mode computes, what the golden model computes, and what the
RTL must compute. Every activation is **unsigned 8-bit**; every weight is **signed 8-bit**.

For conv layer ℓ with input `x` (C×H×W, values 0…255), weights `w` (O×C×3×3, values −127…127),
bias `b` (O values, 32-bit, in accumulator units) and shift `s_ℓ`:

```
acc[o, y, x] = b[o] + Σ_c Σ_{i,j ∈ 0..2}  w[o,c,i,j] · xpad[c, 2y+i, 2x+j]      xpad = x with a zero border of 1
out[o, y, x] = clamp( acc >>> s_ℓ , 0, 255 )                                     >>> = arithmetic shift = floor(acc / 2^s)

gap[c]   = ( Σ_{y,x} conv5[c,y,x] ) >>> s_gap          s_gap = ceil(log2(H5·W5))  →  gap ≤ 255 always
logit[k] = b_fc[k] + Σ_c W_fc[k,c] · gap[c]           no shift, no clamp
margin   = logit[1] − logit[0];     POSITIVE  ⇔  margin > T
```

Why each choice, from the hardware's side:

- **Unsigned activations.** ReLU output is never negative, so an int8 activation wastes its sign
  bit. uint8 doubles activation resolution for free. The product becomes 9-bit-signed × 8-bit,
  which a DSP48E1 (25×18 ports) takes directly.
- **ReLU and saturation are one `clamp`.** An 8-bit output needs saturation anyway; merging them
  is one comparator pair per output.
- **`floor`, not round.** Rounding means adding 2^(s−1) before the shift. Training absorbs that
  constant into the bias, so the hardware drops the adder.
- **32-bit bias in accumulator units.** v0's INT8 biases could not reach the logit scale
  (handoff §3.4). There are only 170 biases, so the storage cost is nothing.
- **`>> s_gap` instead of `/ 49`.** Dividing by 49 needs a divider; dividing by 64 is a wire. The
  mean is scaled by 49/64, but training sees the same operation, so nothing is lost and the golden
  model stays exact.
- **Margin + threshold register.** Positive scaling of logits does not change the sign of
  `margin − T`, so `T` is the one number that sets the operating point (handoff §3.4).

In [ ]:
CH = (1,) + tuple(CFG['CHANNELS'])
def arch_table(img):
    rows, h = [], img
    for i in range(5):
        ho = (h + 2 - 3) // 2 + 1
        rows.append(dict(layer=f'conv{i+1}', in_shape=f'{CH[i]}x{h}x{h}', out_shape=f'{CH[i+1]}x{ho}x{ho}',
                         weights=CH[i+1] * CH[i] * 9, biases=CH[i+1], MACs=ho * ho * CH[i+1] * CH[i] * 9,
                         fmap_bytes=CH[i+1] * ho * ho, linebuf_bytes=2 * (h + 2) * CH[i]))
        h = ho
    rows.append(dict(layer='gap', in_shape=f'{CH[5]}x{h}x{h}', out_shape=f'{CH[5]}', weights=0, biases=0, MACs=0,
                     fmap_bytes=CH[5], linebuf_bytes=0))
    rows.append(dict(layer='fc', in_shape=f'{CH[5]}', out_shape='2', weights=2 * CH[5], biases=2, MACs=2 * CH[5],
                     fmap_bytes=0, linebuf_bytes=0))
    return pd.DataFrame(rows), h

ARCH, FINAL_HW = arch_table(IMG)
GAP_SHIFT = int(math.ceil(math.log2(FINAL_HW * FINAL_HW)))
print(ARCH.to_string(index=False))
print(f'\ntotal parameters {ARCH.weights.sum() + ARCH.biases.sum():,}  ({ARCH.weights.sum():,} int8 weights + {ARCH.biases.sum()} 32-bit biases)')
print(f'total MACs per image {ARCH.MACs.sum():,}')
print(f'final map {FINAL_HW}x{FINAL_HW} -> s_gap = ceil(log2({FINAL_HW*FINAL_HW})) = {GAP_SHIFT}  '
      f'(max gap value {(FINAL_HW*FINAL_HW*255) >> GAP_SHIFT} <= 255)')
for s in (28, 224, 512):
    t, fh = arch_table(s)
    print(f'  @{s:3d}: MACs {t.MACs.sum():>12,}  largest feature map {t.fmap_bytes.max():>9,} B  final map {fh}x{fh}')

## 9. The model, defined directly in integer space

Same philosophy as v0: the trainable parameters **are** the integers. There are no scale
factors to export, so there are none to get wrong.

- `weight` lives in int8 units. After every optimiser step it is projected back into
  [−127, 127]. In hard mode the forward pass uses `round(weight)` with a straight-through
  estimator (gradient flows as if rounding were the identity).
- `bias_act` lives in **activation units** and is multiplied by 2^s to reach accumulator units.
  This keeps bias and weight gradients on similar scales so one learning rate works for both;
  the exported bias is `round(bias_act · 2^s)`.
- `requant` is `floor(acc / 2^s)` (straight-through in hard mode), then `clamp(0, 255)`.
- `forward(..., trace=True)` returns every intermediate tensor under the same names the golden
  model and the RTL trace files use.

**No mixed precision.** Accumulators reach millions; float16 tops out at 65,504. AMP would
silently train a different network from the one being exported.

### Calibrating the shifts
Each `s_ℓ` is chosen **once, before training**, then frozen: run a batch of real training images
through the freshly initialised integer network, take the 99th percentile of the positive
accumulator values at each layer, and pick the power of two that maps it to `ACT_TARGET` (128 —
half of the uint8 range, leaving a bit of headroom). Calibration walks layer by layer, so each
shift sees the requantised output of the layer before. The weights then learn *around* the
fixed shifts, which is what makes the shifts free in hardware.

Calibration also sets each channel's **initial bias to minus its median accumulator value**.
Inputs are uint8 (all positive, mean ~120), so without this a channel whose weights happen to
sum negative is negative at every pixel, ReLU zeroes it, and it never receives a gradient again.
A first smoke run showed conv1 stuck at exactly 50% zeros for this reason.

The table printed below shows, per layer, the fraction of outputs at 0 (ReLU) and at 255
(saturated). Around 40–60% zeros is normal. **A layer that sits near 100% zeros is dead, and one
with more than a few percent at 255 is clipping** — both are the first thing to check if training
stalls.

In [ ]:
def ste(x, fn):
    return x + (fn(x) - x).detach()

BIAS_ACT_MAX = 4096.0

class IntConv3x3(nn.Module):
    def __init__(self, cin, cout, init_std):
        super().__init__()
        w = torch.randn(cout, cin, 3, 3) * init_std
        self.weight = nn.Parameter(w.clamp(-127, 127))
        self.bias_act = nn.Parameter(torch.zeros(cout))
        self.register_buffer('shift', torch.tensor(0, dtype=torch.int64))

    def int_weight(self): return self.weight.detach().clamp(-127, 127).round()
    def int_bias(self):   return (self.bias_act.detach() * 2.0 ** int(self.shift)).round()

    def accumulate(self, x, hard):
        w = self.weight.clamp(-127, 127)
        b = self.bias_act * 2.0 ** int(self.shift)
        if hard:
            w, b = ste(w, torch.round), ste(b, torch.round)
        return F.conv2d(x, w, b, stride=2, padding=1)

class IntFC(nn.Module):
    def __init__(self, cin, cout, init_std):
        super().__init__()
        self.weight = nn.Parameter((torch.randn(cout, cin) * init_std).clamp(-127, 127))
        self.bias_act = nn.Parameter(torch.zeros(cout))
        self.register_buffer('loss_shift', torch.tensor(0, dtype=torch.int64))   # training only, NOT hardware

    def int_weight(self): return self.weight.detach().clamp(-127, 127).round()
    def int_bias(self):   return (self.bias_act.detach() * 2.0 ** int(self.loss_shift)).round()

    def forward(self, g, hard):
        w = self.weight.clamp(-127, 127)
        b = self.bias_act * 2.0 ** int(self.loss_shift)
        if hard:
            w, b = ste(w, torch.round), ste(b, torch.round)
        return F.linear(g, w, b)

def requant(acc, shift, hard):
    y = acc / 2.0 ** int(shift)
    if hard:
        y = ste(y, torch.floor)
    return y.clamp(0, 255)

class FlashV1(nn.Module):
    def __init__(self, channels, img, init_std):
        super().__init__()
        ch = (1,) + tuple(channels)
        self.convs = nn.ModuleList([IntConv3x3(ch[i], ch[i + 1], init_std) for i in range(5)])
        self.fc = IntFC(ch[-1], 2, init_std)
        h = img
        for _ in range(5): h = (h - 1) // 2 + 1
        self.final_hw = h
        self.register_buffer('gap_shift', torch.tensor(int(math.ceil(math.log2(h * h))), dtype=torch.int64))
        self.log_temp = nn.Parameter(torch.zeros(()))   # loss-only temperature; never exported

    def forward(self, x, hard=False, trace=False):
        tr = {}
        for i, L in enumerate(self.convs):
            acc = L.accumulate(x, hard)
            x = requant(acc, L.shift, hard)
            if trace: tr[f'conv{i+1}_acc'], tr[f'conv{i+1}'] = acc, x
        assert x.shape[-1] == self.final_hw, 'input resolution does not match the model'
        gsum = x.sum(dim=(2, 3))
        g = gsum / 2.0 ** int(self.gap_shift)
        if hard: g = ste(g, torch.floor)
        logits = self.fc(g, hard)
        if trace: tr['gap_sum'], tr['gap'], tr['logits'] = gsum, g, logits
        return logits, tr

    @torch.no_grad()
    def project(self):
        for p in [*self.convs, self.fc]:
            p.weight.clamp_(-127, 127); p.bias_act.clamp_(-BIAS_ACT_MAX, BIAS_ACT_MAX)

@torch.no_grad()
def calibrate(model, xb, target):
    x, rows = xb, []
    for i, L in enumerate(model.convs):
        acc = F.conv2d(x, L.int_weight(), None, stride=2, padding=1)
        # centre each channel: uint8 inputs are all positive, so a channel whose weights sum
        # negative would be negative everywhere and dead from step 0. Bias = -median.
        med = acc.transpose(0, 1).reshape(acc.shape[1], -1).median(dim=1).values
        acc = acc - med.view(1, -1, 1, 1)
        pos = acc[acc > 0].flatten()
        if pos.numel() > 1_000_000:
            pos = pos[torch.randint(0, pos.numel(), (1_000_000,), device=pos.device)]
        p99 = torch.quantile(pos.float(), 0.99).item() if pos.numel() else 1.0
        s = max(0, int(round(math.log2(max(p99, 1.0) / target))))
        L.shift.fill_(s)
        L.bias_act.data.copy_((-med / 2.0 ** s).clamp(-BIAS_ACT_MAX, BIAS_ACT_MAX))
        acc = acc + (L.int_bias().view(1, -1, 1, 1) + med.view(1, -1, 1, 1))   # use the rounded bias exactly
        x = torch.clamp(torch.floor(acc / 2.0 ** s), 0, 255)
        rows.append(dict(layer=f'conv{i+1}', acc_p99=f'{p99:,.0f}', shift=s, **{'2^s': 2 ** s},
                         zeros=f'{(x == 0).float().mean().item():.1%}', dead_ch=int((x.amax(dim=(0, 2, 3)) == 0).sum()), sat255=f'{(x == 255).float().mean().item():.2%}'))
    g = torch.floor(x.sum(dim=(2, 3)) / 2.0 ** int(model.gap_shift))
    lg = F.linear(g, model.fc.int_weight())
    ls = max(0, int(round(math.log2(max(lg.std().item(), 1.0) / 2.0))))
    model.fc.loss_shift.fill_(ls)
    print(pd.DataFrame(rows).to_string(index=False))
    print(f'gap shift {int(model.gap_shift)} | fc logit std at init {lg.std().item():,.0f} -> loss_shift {ls} (loss only, not hardware)')

torch.manual_seed(SEED)
model = FlashV1(CFG['CHANNELS'], IMG, CFG['INIT_STD']).to(DEV)
calib_idx = np.random.default_rng(SEED).choice(len(Xtr), min(512, len(Xtr)), replace=False)
calibrate(model, torch.from_numpy(Xtr[calib_idx]).to(DEV).float(), CFG['ACT_TARGET'])
print(f'{sum(p.numel() for p in model.parameters()):,} trainable parameters')

## 10. Train

**Two phases**, as in v0. The soft phase lets the network learn with continuous weights; the hard
phase switches on straight-through rounding so it adapts to being an integer network. The line
`QAT gap` printed at the switch is the accuracy cost of rounding before any adaptation.

**Model selection** uses **validation AUROC of the integer network** (hard mode), in the hard
phase only. AUROC rather than accuracy because it does not depend on a threshold — the threshold
is chosen afterwards, in §13.

**Loss.** Class-weighted cross-entropy (positives are ~22.5%). The logits are divided by
`2^loss_shift` and multiplied by a **learnable temperature** (`T=` in the log) before softmax.
Positive scaling never changes the sign of `margin − T_threshold`, so the hardware never sees it —
but without it the loss sat at ln 2 while the network was already ranking well, because 8-bit
weights alone cannot produce confident softmax outputs at a fixed scale.

**Augmentation** (RSNA only), all applied on the GPU per batch:
- rotation ±7°, scale ±8%, translation ±6% — patient positioning varies between films;
- contrast ±15%, brightness ±15 levels — exposure and processing vary between scanners;
- **no horizontal flip** — the heart sits on the left; a flipped radiograph is anatomically a
  different (rare) patient, and consolidation patterns are not mirror-symmetric;
- the augmented image is **rounded back to integers**, because the hardware only ever sees
  integer pixels.

Epoch time on a T4: roughly seconds at 28×28 and tens of seconds at 224×224 (the whole training
set sits on the GPU as uint8, so there is no loader bottleneck).

In [ ]:
def augment(xb):
    B, dev = xb.shape[0], xb.device
    ang = (torch.rand(B, device=dev) * 2 - 1) * math.radians(7)
    sc  = 1 + (torch.rand(B, device=dev) * 2 - 1) * 0.08
    tr  = (torch.rand(B, 2, device=dev) * 2 - 1) * 0.06
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tr[:, 0]], 1), torch.stack([sin, cos, tr[:, 1]], 1)], 1)
    grid = F.affine_grid(theta, xb.shape, align_corners=False)
    xb = F.grid_sample(xb, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
    con = 1 + (torch.rand(B, 1, 1, 1, device=dev) - 0.5) * 0.30
    bri = (torch.rand(B, 1, 1, 1, device=dev) - 0.5) * 30
    return ((xb - 127.5) * con + 127.5 + bri).round().clamp(0, 255)

@torch.no_grad()
def margins_fast(m, X, hard=True, bs=None):
    """Network margins on the training device (float32). For monitoring only --
    every reported number later comes from the exact golden model."""
    bs = bs or (1024 if IMG <= 28 else 256)
    out = []
    for i in range(0, len(X), bs):
        lg, _ = m(torch.from_numpy(X[i:i + bs]).to(DEV).float(), hard=hard)
        out.append((lg[:, 1] - lg[:, 0]).double().cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def layer_health(m, xb):
    _, tr = m(xb, hard=True, trace=True)
    return ' '.join(f'c{i}:{(tr[f"conv{i}"] == 0).float().mean().item():.0%}/{(tr[f"conv{i}"] == 255).float().mean().item():.1%}'
                    f'/{int((tr[f"conv{i}"].amax(dim=(0, 2, 3)) == 0).sum())}d' for i in range(1, 6))

on_gpu_data = DEV == 'cuda' and Xtr.nbytes < 3e9
Xtr_dev = torch.from_numpy(Xtr).to(DEV) if on_gpu_data else torch.from_numpy(Xtr)
Ytr_dev = torch.from_numpy(Ytr).to(DEV)
n_pos = int(Ytr.sum()); n_neg = len(Ytr) - n_pos
W_CLS = torch.tensor([len(Ytr) / (2 * n_neg), len(Ytr) / (2 * n_pos)], device=DEV)
health_batch = torch.from_numpy(Xva[:64]).to(DEV).float()
print('health columns: conv<i>: %zeros / %saturated / dead channels')
print(f'train {len(Ytr)} (pos {n_pos}) | class weights {W_CLS.tolist()} | data on {"GPU" if on_gpu_data else "CPU"}')

def param_groups(lr):
    body = [p for n, p in model.named_parameters() if n != 'log_temp']
    return [dict(params=body, lr=lr), dict(params=[model.log_temp], lr=0.02)]

E_SOFT, E_HARD = CFG['EPOCHS_SOFT'], CFG['EPOCHS_HARD']
TOT, BS = E_SOFT + E_HARD, CFG['BATCH']
history, best = [], dict(auroc=-1.0, state=None, epoch=None)
opt = torch.optim.Adam(param_groups(CFG['LR_SOFT']))
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(E_SOFT, 1), eta_min=CFG['LR_SOFT'] * 0.1)
gen = torch.Generator(device='cpu').manual_seed(SEED)
t_start = time.time()

for ep in range(TOT):
    hard = ep >= E_SOFT
    if ep == E_SOFT:
        soft_auc = roc_auc_score(Yva, margins_fast(model, Xva, hard=False))
        hard_auc = roc_auc_score(Yva, margins_fast(model, Xva))
        print(f'--- switching on integer rounding | val AUROC soft {soft_auc:.4f} -> hard {hard_auc:.4f}  (QAT gap {soft_auc-hard_auc:+.4f}) ---')
        opt = torch.optim.Adam(param_groups(CFG['LR_HARD']))
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(E_HARD, 1), eta_min=CFG['LR_HARD'] * 0.1)

    model.train(); t0 = time.time(); tot_loss = 0.0
    perm = torch.randperm(len(Ytr), generator=gen)
    for i in range(0, len(perm), BS):
        idx = perm[i:i + BS]
        xb = Xtr_dev[idx.to(Xtr_dev.device)].to(DEV, non_blocking=True).float()
        yb = Ytr_dev[idx.to(DEV)]
        if CFG['AUG']: xb = augment(xb)
        logits, _ = model(xb, hard=hard)
        loss = F.cross_entropy(logits / 2.0 ** int(model.fc.loss_shift) * model.log_temp.exp(), yb, weight=W_CLS)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); model.project()
        tot_loss += loss.item() * len(idx)
    sch.step()

    val_auc = roc_auc_score(Yva, margins_fast(model, Xva))
    tag = ''
    if hard and val_auc > best['auroc']:
        best = dict(auroc=val_auc, state=copy.deepcopy(model.state_dict()), epoch=ep); tag = ' <- best'
    history.append(dict(epoch=ep, loss=tot_loss / len(Ytr), val_auroc=val_auc, hard=hard))
    if ep % 5 == 0 or tag or ep >= TOT - 2 or ep == E_SOFT - 1:
        print(f'ep {ep:3d}/{TOT} {"hard" if hard else "soft"} loss {tot_loss/len(Ytr):.4f} T={model.log_temp.exp().item():5.2f} | int val AUROC {val_auc:.4f}'
              f' | {time.time()-t0:5.1f}s | zero/sat {layer_health(model, health_batch)}{tag}')

model.load_state_dict(best['state'])
print(f'\ntraining {time.time()-t_start:.0f}s | best integer val AUROC {best["auroc"]:.4f} at epoch {best["epoch"]}')

H = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(H.epoch, H.loss); ax[0].set_title('training loss'); ax[0].axvline(E_SOFT - 0.5, ls='--', c='gray')
ax[1].plot(H.epoch, H.val_auroc); ax[1].set_title('validation AUROC (integer network)'); ax[1].axvline(E_SOFT - 0.5, ls='--', c='gray')
ax[1].axvline(best['epoch'], c='green', lw=0.8)
for a in ax: a.set_xlabel('epoch  (dashed = integer rounding on, green = selected)')
plt.tight_layout(); plt.savefig(EXPORT / 'report' / 'training.png', dpi=90); plt.show()
torch.save(dict(state=best['state'], cfg=CFG), EXPORT / 'model_state.pt')

## 11. Export: weights, biases, and the layer-descriptor ROM

### Memory files
| File | Width | Order |
|---|---|---|
| `weights.mem` | 8-bit two's complement, 2 hex digits/line | all layers concatenated; conv `[out, in, ky, kx]`, fc `[out, in]` |
| `bias.mem` | 32-bit two's complement, 8 hex digits/line | all layers concatenated |

### The layer-descriptor table — why it decouples the network from the Verilog
In v0 each layer was a hand-written module with its sizes baked into the RTL. In V1 the RTL is
**one conv engine plus a sequencer**. The sequencer reads one 128-bit word per layer from a ROM,
loads those fields into counters and address offsets, runs the engine, and moves to the next word.
Everything that describes *the network* lives in that ROM:

| Field | Bits | Meaning |
|---|---|---|
| `op` | 4 | 1 = CONV3X3, 2 = GAP, 3 = FC. **0 is deliberately unused**: an unprogrammed (all-zero) word is detectably invalid instead of silently decoding as a real layer |
| `kernel`, `stride`, `pad` | 4 each | 3 / 2 / 1 for every conv here, but carried so the engine can be generalised without a new format |
| `in_c`, `out_c` | 12 each | loop bounds for channels (≤ 4095) |
| `in_h`, `in_w`, `out_h`, `out_w` | 12 each | loop bounds for rows/columns — 12 bits so 512 and beyond need no format change |
| `shift` | 6 | `s_ℓ` for convs, `s_gap` for GAP, 0 for FC |
| `w_base` | 20 | first index of this layer's weights in `weights.mem` |
| `b_base` | 10 | first index of this layer's biases in `bias.mem` |

That is 124 bits, packed MSB-first from bit 127, 4 reserved. Changing channel counts, depth or
resolution becomes: re-run this notebook, re-synthesise with the new `.mem` files. No Verilog edit.

**The field positions themselves are generated** into `layer_table.vh` as `` `define``s, so the
notebook and the RTL can never disagree about where a field sits. The same table is written as
`layer_table.mem` (the ROM image) and `layer_table.json` (read by the golden model).

In [ ]:
OPCODE = {'CONV3X3': 1, 'GAP': 2, 'FC': 3}
LT_FIELDS = [('op', 4), ('kernel', 4), ('stride', 4), ('pad', 4), ('in_c', 12), ('out_c', 12),
             ('in_h', 12), ('in_w', 12), ('out_h', 12), ('out_w', 12), ('shift', 6), ('w_base', 20), ('b_base', 10)]
LT_WIDTH = 128
assert sum(b for _, b in LT_FIELDS) <= LT_WIDTH

def pack_layer(L):
    word, pos = 0, LT_WIDTH
    for f, bits in LT_FIELDS:
        v = OPCODE[L['op']] if f == 'op' else int(L[f])
        assert 0 <= v < (1 << bits), f'{L["name"]}.{f}={v} does not fit {bits} bits'
        pos -= bits; word |= v << pos
    return word

model_cpu = copy.deepcopy(model).cpu()
layers, wparts, bparts = [], [], []
h, cin = IMG, 1
for i, L in enumerate(model_cpu.convs):
    W = L.int_weight().numpy().astype(np.int64); B = L.int_bias().numpy().astype(np.int64)
    ho = (h + 2 - 3) // 2 + 1
    layers.append(dict(name=f'conv{i+1}', op='CONV3X3', kernel=3, stride=2, pad=1, in_c=cin, out_c=W.shape[0],
                       in_h=h, in_w=h, out_h=ho, out_w=ho, shift=int(L.shift),
                       w_base=sum(p.size for p in wparts), b_base=sum(p.size for p in bparts), n_w=int(W.size), n_b=int(B.size)))
    wparts.append(W.reshape(-1)); bparts.append(B.reshape(-1)); h, cin = ho, W.shape[0]
layers.append(dict(name='gap', op='GAP', kernel=0, stride=0, pad=0, in_c=cin, out_c=cin, in_h=h, in_w=h, out_h=1, out_w=1,
                   shift=int(model_cpu.gap_shift), w_base=0, b_base=0, n_w=0, n_b=0))
W = model_cpu.fc.int_weight().numpy().astype(np.int64); B = model_cpu.fc.int_bias().numpy().astype(np.int64)
layers.append(dict(name='fc', op='FC', kernel=1, stride=1, pad=0, in_c=cin, out_c=2, in_h=1, in_w=1, out_h=1, out_w=1,
                   shift=0, w_base=sum(p.size for p in wparts), b_base=sum(p.size for p in bparts), n_w=int(W.size), n_b=int(B.size)))
wparts.append(W.reshape(-1)); bparts.append(B.reshape(-1))
WFLAT, BFLAT = np.concatenate(wparts), np.concatenate(bparts)
assert WFLAT.min() >= -127 and WFLAT.max() <= 127
assert BFLAT.min() >= -(1 << 31) and BFLAT.max() < (1 << 31), 'bias exceeds 32 bits'

(EXPORT / 'weights.mem').write_text(''.join(f'{int(v) & 0xFF:02x}\n' for v in WFLAT))
(EXPORT / 'bias.mem').write_text(''.join(f'{int(v) & 0xFFFFFFFF:08x}\n' for v in BFLAT))
(EXPORT / 'layer_table.mem').write_text(''.join(f'{pack_layer(L):032x}\n' for L in layers))
(EXPORT / 'layer_table.json').write_text(json.dumps(dict(fields=LT_FIELDS, opcodes=OPCODE, word_width=LT_WIDTH,
                                                         img=IMG, layers=layers), indent=1))
vh = ['// AUTO-GENERATED by ProjectFlash_V1.ipynb -- do not edit by hand',
      f'// stage {STAGE}{" (SMOKE TEST)" if SMOKE_TEST else ""}, input {IMG}x{IMG}', '',
      f'`define LT_WORD_W {LT_WIDTH}']
pos = LT_WIDTH
for f, bits in LT_FIELDS:
    vh += [f'`define LT_{f.upper()}_MSB {pos-1}', f'`define LT_{f.upper()}_LSB {pos-bits}']; pos -= bits
vh += [f'`define LT_OP_{k} 4\'d{v}' for k, v in OPCODE.items()]
vh += ['', f'`define FLASH_N_LAYERS {len(layers)}', f'`define FLASH_IMG_H {IMG}', f'`define FLASH_IMG_W {IMG}',
       f'`define FLASH_N_WEIGHTS {WFLAT.size}', f'`define FLASH_N_BIASES {BFLAT.size}', '', '// table contents:']
vh += [f'//  {i}: {L["name"]:5s} {L["op"]:7s} {L["in_c"]:2d}x{L["in_h"]:3d}x{L["in_w"]:3d} -> {L["out_c"]:2d}x{L["out_h"]:3d}x{L["out_w"]:3d}'
       f'  shift {L["shift"]:2d}  w_base {L["w_base"]:6d}  b_base {L["b_base"]:3d}  word {pack_layer(L):032x}' for i, L in enumerate(layers)]
(EXPORT / 'layer_table.vh').write_text('\n'.join(vh) + '\n')

print(pd.DataFrame(layers).drop(columns=['kernel', 'stride', 'pad']).to_string(index=False))
print(f'\nweights.mem {WFLAT.size} values [{WFLAT.min()}, {WFLAT.max()}] | bias.mem {BFLAT.size} values [{BFLAT.min()}, {BFLAT.max()}]')
print((EXPORT / 'layer_table.vh').read_text())

## 12. The golden model, and proof that it equals the trained network

The golden model is written to its own file and **reads only the exported `.mem` and `.json`
files** — not the PyTorch model. So when it agrees with PyTorch, that proves two things at once:
the arithmetic contract is implemented correctly, *and* the export (hex encoding, two's
complement, ordering, base offsets) round-trips. It is pure NumPy `int64`: no floating point
anywhere, so there is nothing to be approximately right about.

### The two checks
1. **Every intermediate tensor** — all five conv accumulators, all five conv outputs, the GAP sum,
   the GAP output and the logits — on every verification image.
2. **Logits on the entire test set** (`EXACT_CHECK_FULL_TEST`).

The PyTorch side runs in **float64 on the CPU**. Why not the GPU: cuDNN may pick convolution
algorithms (FFT, Winograd) that are not exact even when every input is an integer. On the CPU in
float64, every value here is an integer below 2⁵³, so the arithmetic is exact regardless of
summation order. Training used float32 on the GPU, which is fine — the export is what counts, and
this cell checks the export.

### Choosing the verification images
A balanced slice of the **test** set — half positive, half negative, shuffled with a fixed seed —
with **no confidence filter**. Some of these the model gets wrong, on purpose: the RTL must
reproduce the golden model including its mistakes.

In [ ]:
%%writefile flash_v1_tools/golden_model_v1.py
"""Project FLASH V1 -- standalone golden model.

Pure NumPy int64. Reads ONLY exported files (layer_table.json, weights.mem, bias.mem),
so agreement with the training framework proves the export as well as the arithmetic.
This is the reference the RTL must match bit for bit.

    from golden_model_v1 import GoldenModel
    g = GoldenModel('export/v1_2')
    logits, trace = g.run(images_uint8_N1HW, keep=True)
"""
import json
from pathlib import Path
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view


def read_hex(path, bits, signed):
    v = np.array([int(t, 16) for t in Path(path).read_text().split()], dtype=np.int64)
    if signed:
        v = np.where(v >= (1 << (bits - 1)), v - (1 << bits), v)
    return v


class GoldenModel:
    def __init__(self, export_dir):
        d = Path(export_dir)
        spec = json.loads((d / 'layer_table.json').read_text())
        self.layers, self.img = spec['layers'], spec['img']
        w = read_hex(d / 'weights.mem', 8, True)
        b = read_hex(d / 'bias.mem', 32, True)
        for L in self.layers:
            if L['op'] == 'CONV3X3':
                L['W'] = w[L['w_base']:L['w_base'] + L['n_w']].reshape(L['out_c'], L['in_c'], 3, 3)
                L['B'] = b[L['b_base']:L['b_base'] + L['n_b']]
            elif L['op'] == 'FC':
                L['W'] = w[L['w_base']:L['w_base'] + L['n_w']].reshape(L['out_c'], L['in_c'])
                L['B'] = b[L['b_base']:L['b_base'] + L['n_b']]

    @staticmethod
    def conv3x3_acc(x, W, B, stride, pad):
        """acc[n,o,y,x] = B[o] + sum_{c,i,j} W[o,c,i,j] * xpad[n,c,stride*y+i,stride*x+j]"""
        n, c, h, w = x.shape
        xp = np.zeros((n, c, h + 2 * pad, w + 2 * pad), dtype=np.int64)
        xp[:, :, pad:pad + h, pad:pad + w] = x
        ho, wo = (h + 2 * pad - 3) // stride + 1, (w + 2 * pad - 3) // stride + 1
        win = sliding_window_view(xp, (3, 3), axis=(2, 3))[:, :, ::stride, ::stride][:, :, :ho, :wo]
        return np.einsum('nchwij,ocij->nohw', win, W, optimize=True) + B[None, :, None, None]

    def run(self, x_u8, keep=False, acc_stats=None):
        """x_u8: (N,1,H,W) uint8. Returns (logits (N,2) int64, trace dict)."""
        a = np.asarray(x_u8).astype(np.int64)
        assert a.shape[1:] == (1, self.img, self.img), f'expected (N,1,{self.img},{self.img})'
        trace = {}

        def stat(name, v):
            if acc_stats is not None:
                lo, hi = int(v.min()), int(v.max())
                o = acc_stats.get(name, (lo, hi))
                acc_stats[name] = (min(o[0], lo), max(o[1], hi))

        for L in self.layers:
            nm = L['name']
            if L['op'] == 'CONV3X3':
                acc = self.conv3x3_acc(a, L['W'], L['B'], L['stride'], L['pad'])
                a = np.clip(acc >> L['shift'], 0, 255)          # >> on int64 = arithmetic = floor
                stat(nm + '_acc', acc)
                if keep: trace[nm + '_acc'], trace[nm] = acc, a
            elif L['op'] == 'GAP':
                s = a.sum(axis=(2, 3))
                a = s >> L['shift']
                stat('gap_sum', s)
                if keep: trace['gap_sum'], trace['gap'] = s, a
            elif L['op'] == 'FC':
                a = a @ L['W'].T + L['B'][None, :]
                stat('logits', a); stat('margin', a[:, 1] - a[:, 0])
                if keep: trace['logits'] = a
        return a, trace


def decide(logits, threshold):
    """POSITIVE iff logit1 - logit0 > threshold."""
    return ((logits[:, 1] - logits[:, 0]) > threshold).astype(np.int64)

In [ ]:
import golden_model_v1 as gm
importlib.reload(gm)
G = gm.GoldenModel(EXPORT)

def golden_all(X, acc_stats, bs=32):
    out = []
    for i in range(0, len(X), bs):
        lg, _ = G.run(X[i:i + bs], acc_stats=acc_stats)
        out.append(lg)
    return np.concatenate(out)

ACC_STATS = {}
t = time.time(); LG_VA = golden_all(Xva, ACC_STATS); LG_TE = golden_all(Xte, ACC_STATS)
M_VA, M_TE = LG_VA[:, 1] - LG_VA[:, 0], LG_TE[:, 1] - LG_TE[:, 0]
print(f'golden model: {len(Xva)} val + {len(Xte)} test images in {time.time()-t:.1f}s')

# ---- verification selection: balanced slice of test, no confidence filter ----
N = min(CFG['N_VERIFY'], 2 * min((Yte == 0).sum(), (Yte == 1).sum()))
rng = np.random.default_rng(SEED)
sel = np.concatenate([rng.permutation(np.where(Yte == 0)[0])[:N // 2], rng.permutation(np.where(Yte == 1)[0])[:N - N // 2]])
sel = rng.permutation(sel)
print(f'verification set: {len(sel)} test images ({(Yte[sel]==0).sum()} negative, {(Yte[sel]==1).sum()} positive)')

# ---- check 1: every intermediate tensor on every verification image ----
m64 = copy.deepcopy(model).cpu().double().eval()
torch.set_num_threads(max(1, os.cpu_count() or 1))
TRACE_NAMES = [f'conv{i}{s}' for i in range(1, 6) for s in ('_acc', '')] + ['gap_sum', 'gap', 'logits']
mism = {k: 0 for k in TRACE_NAMES}; elems = {k: 0 for k in TRACE_NAMES}
t = time.time()
with torch.no_grad():
    for i in range(0, len(sel), 16):
        xb = Xte[sel[i:i + 16]]
        _, gtr = G.run(xb, keep=True)
        _, ttr = m64(torch.from_numpy(xb).double(), hard=True, trace=True)
        for k in TRACE_NAMES:
            tv = ttr[k].numpy()
            assert np.all(tv == np.round(tv)), f'{k}: torch produced a non-integer'
            mism[k] += int((tv.astype(np.int64) != gtr[k]).sum()); elems[k] += gtr[k].size
print(f'\nCHECK 1 -- intermediate tensors, {len(sel)} images ({time.time()-t:.1f}s)')
for k in TRACE_NAMES:
    print(f'  {k:10s} {mism[k]:>8d} mismatches / {elems[k]:>12,d} values')
CHECK1_OK = sum(mism.values()) == 0

# ---- check 2: logits on the whole test set ----
t = time.time()
chk_idx = np.arange(len(Xte)) if CFG['EXACT_CHECK_FULL_TEST'] else sel
bad = 0
with torch.no_grad():
    for i in range(0, len(chk_idx), 64):
        lg, _ = m64(torch.from_numpy(Xte[chk_idx[i:i + 64]]).double(), hard=True)
        bad += int((lg.numpy().astype(np.int64) != LG_TE[chk_idx[i:i + 64]]).any(axis=1).sum())
CHECK2 = (len(chk_idx) - bad, len(chk_idx))
print(f'\nCHECK 2 -- logits, torch(float64, hard) vs golden: {CHECK2[0]} / {CHECK2[1]} images bit-exact ({time.time()-t:.1f}s)')
assert CHECK1_OK and bad == 0, 'GOLDEN MODEL DIVERGES FROM THE TRAINED NETWORK -- do not export'
print('\n*** golden model == trained integer network, bit for bit ***')

## 13. Clinical evaluation — every number here is what the FPGA will compute

All metrics below are computed from the **golden model's integer margins**, which §12 proved equal
to the trained network and which the RTL will be required to match. There is no separate
"software accuracy" to confuse with hardware accuracy.

Keep two numbers separate, as in v0: *"the RTL is bit-exact"* (a verification result) and
*"the model has AUROC X"* (a model-quality result). This section is the second.

### Choosing the operating point `T` — on validation, then frozen
A screening tool is usually run at high sensitivity: a missed pneumonia costs more than a false
alarm that a radiologist dismisses. So the default `T` is the **largest** integer threshold whose
**validation** sensitivity is at least `TARGET_SENS` — largest, because among all thresholds that
reach the target it gives the best specificity. It is then applied, unchanged, to the test set.
Test sensitivity will land near the target, not exactly on it; the 95% confidence intervals show
how much of that is sampling noise.

Also reported: the Youden-optimal threshold (max sensitivity + specificity − 1, also chosen on
validation), and **`T = 0`, which is plain argmax — what v0 did**, for comparison.

### Subgroups
- **Three-way class.** "Opacity vs Normal" is the easy question; "Opacity vs *No Lung Opacity /
  Not Normal*" is the hard one (effusions, nodules, other findings that are abnormal but not
  pneumonia). Reporting only the combined AUROC hides this.
- **AP vs PA view.** If the model's AUROC collapses *within* each view compared with overall, it
  was leaning on the view-position shortcut measured in §7.
- **Sex.** A basic fairness check; large gaps need explaining before any clinical use.

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0: return (float('nan'),) * 3
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, c - h, c + h

def op_point(y, m, T):
    p = m > T
    tp, fn = int((p & (y == 1)).sum()), int((~p & (y == 1)).sum())
    tn, fp_ = int((~p & (y == 0)).sum()), int((p & (y == 0)).sum())
    return dict(sens=wilson(tp, tp + fn), spec=wilson(tn, tn + fp_),
                ppv=tp / (tp + fp_) if tp + fp_ else float('nan'), npv=tn / (tn + fn) if tn + fn else float('nan'),
                tp=tp, fn=fn, tn=tn, fp=fp_)

def threshold_for_sensitivity(y, m, target):
    pos = np.sort(m[y == 1]); need = int(math.ceil(target * len(pos)))
    return int(pos[len(pos) - need] - 1)          # largest T with count(m_pos > T) >= need

def youden_threshold(y, m):
    fpr, tpr, thr = roc_curve(y, m)
    k = int(np.argmax((tpr - fpr)[1:])) + 1          # skip the +inf threshold
    return int(thr[k]) - 1                           # m >= thr  <=>  m > thr-1 for integers

def auroc_ci(y, m, B, seed=0):
    r = np.random.default_rng(seed); vals = []
    for _ in range(B):
        i = r.integers(0, len(y), len(y))
        if y[i].min() != y[i].max(): vals.append(roc_auc_score(y[i], m[i]))
    return roc_auc_score(y, m), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

VAL_AUC = auroc_ci(Yva, M_VA, CFG['BOOTSTRAP'])
TEST_AUC = auroc_ci(Yte, M_TE, CFG['BOOTSTRAP'])
T_SENS = threshold_for_sensitivity(Yva, M_VA, CFG['TARGET_SENS'])
T_YOUDEN = youden_threshold(Yva, M_VA)
DEFAULT_T = T_SENS

f3 = lambda t: f'{t[0]:.1%} [{t[1]:.1%}, {t[2]:.1%}]'
print(f'AUROC  val  {VAL_AUC[0]:.4f}  95% CI [{VAL_AUC[1]:.4f}, {VAL_AUC[2]:.4f}]')
print(f'AUROC  test {TEST_AUC[0]:.4f}  95% CI [{TEST_AUC[1]:.4f}, {TEST_AUC[2]:.4f}]')
if SHORTCUT_AUROC is not None:
    print(f'       (view-position-only AUROC on test was {SHORTCUT_AUROC:.3f})')

OPS = {}
rows = []
for name, T in [(f'sens>={CFG["TARGET_SENS"]:.0%} on val (DEFAULT)', T_SENS), ('Youden on val', T_YOUDEN), ('argmax, T=0 (v0)', 0)]:
    v, te = op_point(Yva, M_VA, T), op_point(Yte, M_TE, T)
    OPS[name] = dict(T=T, val=v, test=te)
    rows.append({'operating point': name, 'T': T, 'val sens': f'{v["sens"][0]:.1%}', 'val spec': f'{v["spec"][0]:.1%}',
                 'TEST sens [95% CI]': f3(te['sens']), 'TEST spec [95% CI]': f3(te['spec']),
                 'test PPV': f'{te["ppv"]:.1%}', 'test NPV': f'{te["npv"]:.1%}'})
print('\n' + pd.DataFrame(rows).to_string(index=False))
d = OPS[f'sens>={CFG["TARGET_SENS"]:.0%} on val (DEFAULT)']['test']
print(f'\ndefault operating point on test: TP {d["tp"]}  FN {d["fn"]}  TN {d["tn"]}  FP {d["fp"]}  '
      f'(PPV/NPV at this test prevalence of {Yte.mean():.1%})')

# ---- subgroups on test, at the default threshold ----
sub = []
def add_sub(label, mask, y=None, mm=None):
    y = Yte[mask] if y is None else y; mm = M_TE[mask] if mm is None else mm
    if len(np.unique(y)) < 2 or len(y) < 10: return
    o = op_point(y, mm, DEFAULT_T)
    sub.append(dict(subgroup=label, n=len(y), positives=int(y.sum()), AUROC=f'{roc_auc_score(y, mm):.4f}',
                    sens=f'{o["sens"][0]:.1%}', spec=f'{o["spec"][0]:.1%}'))
add_sub('all test', np.ones(len(Yte), bool))
c3 = mte['class3'].values
for neg in ['Normal', 'No Lung Opacity / Not Normal']:
    mk = (c3 == 'Lung Opacity') | (c3 == neg)
    add_sub(f'Opacity vs {neg}', mk)
if CFG['DATASET'] == 'rsna':
    for v in ['AP', 'PA']: add_sub(f'view {v}', mte['view'].values == v)
    for s in ['M', 'F']:   add_sub(f'sex {s}', mte['sex'].values == s)
SUBGROUPS = pd.DataFrame(sub)
print('\nsubgroups (test, default threshold):\n' + SUBGROUPS.to_string(index=False))

fpr, tpr, _ = roc_curve(Yte, M_TE)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f'test AUROC {TEST_AUC[0]:.3f}')
for name, st, mk in [(f'sens>={CFG["TARGET_SENS"]:.0%} on val (DEFAULT)', 'o', 'default T'), ('Youden on val', 's', 'Youden T'), ('argmax, T=0 (v0)', 'x', 'argmax')]:
    te = OPS[name]['test']; ax.plot(1 - te['spec'][0], te['sens'][0], st, ms=8, label=f'{mk} (T={OPS[name]["T"]})')
ax.plot([0, 1], [0, 1], ls=':', c='gray'); ax.set_xlabel('1 - specificity'); ax.set_ylabel('sensitivity')
ax.set_title(f'{STAGE} integer model, test set'); ax.legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.savefig(EXPORT / 'report' / 'roc_test.png', dpi=90); plt.show()

## 14. Hardware sizing from the trained weights

### Accumulator widths — three bounds, and which one to build
For each layer the cell prints three ranges:

| Bound | How it is computed | Safe for… |
|---|---|---|
| **generic** | `fan_in · 255 · 127 + max|b|` — assumes every weight is ±127 | anything, but wasteful |
| **guaranteed** | per output channel: `Σ max(w,0)·255 + b` and `Σ min(w,0)·255 + b`, using the **actual trained weights** | **any** uint8 input image, ever |
| **measured** | min/max seen by the golden model over the whole val + test set | only images like these |

The handoff (§3.5) said to size from measured ranges. I recommend the **guaranteed** bound instead,
and want to flag that as a deliberate change. Measured ranges come from this dataset's scanners;
the board will see images from others, and an accumulator overflow is a *silent wrong answer* —
the worst failure mode for a medical device. The guaranteed bound uses the real weights, not ±127,
so it is still far tighter than the generic bound; the `cost` column shows how many bits it adds
over the measured range. Since one engine is reused across layers, its accumulator is the widest
guaranteed conv width.

### Memory
Layer-sequential execution needs a layer's input and output buffers alive at the same time, so
the feature-map requirement is the largest **consecutive pair**, not the sum. BRAM36 = 36 Kbit;
the 7Z020 has 140. Treat the tile count as a lower bound — Vivado maps memories in power-of-two
depths and synthesis reports the real number.

In [ ]:
def sbits(lo, hi):
    n = 1
    while not (-(1 << (n - 1)) <= lo and hi <= (1 << (n - 1)) - 1): n += 1
    return n

HW = []
for L in G.layers:
    if L['op'] == 'CONV3X3':
        W, B = L['W'], L['B']
        gen = L['in_c'] * 9 * 255 * 127 + int(np.abs(B).max())
        g_hi = int((np.clip(W, 0, None).sum(axis=(1, 2, 3)) * 255 + B).max())
        g_lo = int((np.clip(W, None, 0).sum(axis=(1, 2, 3)) * 255 + B).min())
        mlo, mhi = ACC_STATS[L['name'] + '_acc']
        HW.append(dict(signal=L['name'] + ' acc', generic_bits=sbits(-gen, gen), guaranteed=f'[{g_lo:,}, {g_hi:,}]',
                       guaranteed_bits=sbits(g_lo, g_hi), measured=f'[{mlo:,}, {mhi:,}]', measured_bits=sbits(mlo, mhi)))
    elif L['op'] == 'GAP':
        GMAX = (L['in_h'] * L['in_w'] * 255) >> L['shift']
        mlo, mhi = ACC_STATS['gap_sum']; top = L['in_h'] * L['in_w'] * 255
        HW.append(dict(signal='gap sum (unsigned)', generic_bits=top.bit_length(), guaranteed=f'[0, {top:,}]',
                       guaranteed_bits=top.bit_length(), measured=f'[{mlo:,}, {mhi:,}]', measured_bits=max(mhi, 1).bit_length()))
    elif L['op'] == 'FC':
        W, B = L['W'], L['B']
        gen = L['in_c'] * 255 * 127 + int(np.abs(B).max())
        l_hi = int((np.clip(W, 0, None) * GMAX).sum(axis=1).max() + B.max())
        l_lo = int((np.clip(W, None, 0) * GMAX).sum(axis=1).min() + B.min())
        D, dB = W[1] - W[0], int(B[1] - B[0])
        m_hi, m_lo = int(np.clip(D, 0, None).sum() * GMAX + dB), int(np.clip(D, None, 0).sum() * GMAX + dB)
        for sig, glo, ghi, gb in [('logit', l_lo, l_hi, sbits(-gen, gen)), ('margin', m_lo, m_hi, sbits(-2 * gen, 2 * gen))]:
            mlo, mhi = ACC_STATS['logits' if sig == 'logit' else 'margin']
            HW.append(dict(signal=sig, generic_bits=gb, guaranteed=f'[{glo:,}, {ghi:,}]', guaranteed_bits=sbits(glo, ghi),
                           measured=f'[{mlo:,}, {mhi:,}]', measured_bits=sbits(mlo, mhi)))
HWT = pd.DataFrame(HW); HWT['cost'] = HWT.guaranteed_bits - HWT.measured_bits
print(HWT.to_string(index=False))
ENGINE_ACC_BITS = int(HWT[HWT.signal.str.startswith('conv')].guaranteed_bits.max())
MARGIN_BITS = int(HWT[HWT.signal == 'margin'].guaranteed_bits.iloc[0])
print(f'\n-> shared conv engine accumulator: {ENGINE_ACC_BITS} bits (widest guaranteed conv range)')
print(f'-> margin / threshold register:   {MARGIN_BITS} bits; default T = {DEFAULT_T} must fit: '
      f'{"OK" if -(1 << (MARGIN_BITS-1)) <= DEFAULT_T < (1 << (MARGIN_BITS-1)) else "WIDEN"}')
print(f'   bias values need {sbits(int(BFLAT.min()), int(BFLAT.max()))} bits (stored as 32)')

BRAM36 = 36 * 1024
fm = [IMG * IMG] + [L['out_c'] * L['out_h'] * L['out_w'] for L in G.layers if L['op'] == 'CONV3X3']
pair = max(fm[i] + fm[i + 1] for i in range(len(fm) - 1))
wbytes, bbytes = WFLAT.size, 4 * BFLAT.size
tiles = lambda nbytes: math.ceil(nbytes * 8 / BRAM36)
print(f'\nmemory @ {IMG}x{IMG}:')
print(f'  input image                 {fm[0]:>9,} B')
print(f'  largest feature map         {max(fm[1:]):>9,} B')
print(f'  largest in+out pair         {pair:>9,} B  -> >= {tiles(pair)} BRAM36')
print(f'  weights (int8)              {wbytes:>9,} B  -> >= {tiles(wbytes)} BRAM36')
print(f'  biases (32-bit)             {bbytes:>9,} B  -> >= {tiles(bbytes)} BRAM36')
print(f'  total                                   >= {tiles(pair)+tiles(wbytes)+tiles(bbytes)} of 140 BRAM36 on the 7Z020')
print(f'  MACs per image              {ARCH.MACs.sum():>9,}')

## 15. Verification vectors for the RTL sweep

Written to `vectors/`:

| File | Content |
|---|---|
| `img_<k>.mem` | `IMG·IMG` lines, one uint8 pixel each, row-major, **unpadded** |
| `exp_logit0.mem`, `exp_logit1.mem`, `exp_margin.mem` | 32-bit two's complement, one line per image |
| `exp_argmax.mem` | `logit1 > logit0` — v0's decision |
| `exp_decision.mem` | `margin > DEFAULT_T` — V1's decision |
| `gt_label.mem` | ground truth, **for reference only** — the pass criterion is the logits |
| `trace/img<k>_<layer>.mem` | first `N_TRACE` images: every conv output and the GAP output, uint8, `[c, y, x]` order |
| `trace/img<k>_<layer>_acc.mem` | first `N_ACC` images: raw conv accumulators and GAP sum, 32-bit |
| `sim_config.vh` | image count, resolution, default threshold, recommended widths |

**Why images are unpadded now.** v0 streamed a 30×30 zero-padded frame so the line buffer never
saw a border. In V1, layers 2–5 read their input from BRAM, where there is no caller to pad it,
so padding must be the engine's job for every layer — including the first. Treating all five
layers identically is simpler than special-casing conv1.

**How to use the traces.** If the logit sweep fails, compare `conv1` first; the first layer that
differs is the broken one. If its `_acc` file matches but its output doesn't, the bug is in the
shift/clamp stage; if `_acc` already differs, it is in window alignment or the multiply-accumulate.
That is the v0 lesson — localise by layer, then by stage — built into the file set.

**Window alignment, stated once for the RTL:** output pixel `(y, x)` of a stride-2 layer is centred
on input pixel `(2y, 2x)` and covers input rows `2y−1 … 2y+1` and columns `2x−1 … 2x+1`, where
index −1 (and index `H` on odd-sized inputs) is zero padding.

In [ ]:
VEC = EXPORT / 'vectors'; TRC = VEC / 'trace'
for old in list(VEC.glob('*.mem')) + list(TRC.glob('*.mem')): old.unlink()
h8 = lambda arr: ''.join(f'{int(v) & 0xFF:02x}\n' for v in np.asarray(arr).reshape(-1))
h32 = lambda arr: ''.join(f'{int(v) & 0xFFFFFFFF:08x}\n' for v in np.asarray(arr).reshape(-1))

LG_SEL = LG_TE[sel]; MG_SEL = LG_SEL[:, 1] - LG_SEL[:, 0]
for k, i in enumerate(sel):
    (VEC / f'img_{k}.mem').write_text(h8(Xte[i, 0]))
(VEC / 'exp_logit0.mem').write_text(h32(LG_SEL[:, 0]))
(VEC / 'exp_logit1.mem').write_text(h32(LG_SEL[:, 1]))
(VEC / 'exp_margin.mem').write_text(h32(MG_SEL))
(VEC / 'exp_argmax.mem').write_text(h8((LG_SEL[:, 1] > LG_SEL[:, 0]).astype(int)))
(VEC / 'exp_decision.mem').write_text(h8((MG_SEL > DEFAULT_T).astype(int)))
(VEC / 'gt_label.mem').write_text(h8(Yte[sel]))

for k in range(min(CFG['N_TRACE'], len(sel))):
    _, tr = G.run(Xte[sel[k:k + 1]], keep=True)
    for nm in [f'conv{i}' for i in range(1, 6)] + ['gap']:
        (TRC / f'img{k}_{nm}.mem').write_text(h8(tr[nm][0]))
    if k < CFG['N_ACC']:
        for nm in [f'conv{i}_acc' for i in range(1, 6)] + ['gap_sum']:
            (TRC / f'img{k}_{nm.replace("_acc", "")}_acc.mem').write_text(h32(tr[nm][0]))

(EXPORT / 'sim_config.vh').write_text('\n'.join([
    '// AUTO-GENERATED by ProjectFlash_V1.ipynb', f'`define FLASH_STAGE "{STAGE}"',
    f'`define N_IMAGES {len(sel)}', f'`define IMG_H {IMG}', f'`define IMG_W {IMG}', f'`define N_PIXELS {IMG*IMG}',
    f'`define N_TRACE {min(CFG["N_TRACE"], len(sel))}', f'`define N_ACC {min(CFG["N_ACC"], len(sel))}',
    f'`define DEFAULT_THRESHOLD 32\'h{DEFAULT_T & 0xFFFFFFFF:08x}   // {DEFAULT_T}',
    f'`define REC_ACC_BITS {ENGINE_ACC_BITS}', f'`define REC_MARGIN_BITS {MARGIN_BITS}', '']))

agree = int(((MG_SEL > DEFAULT_T).astype(int) == Yte[sel]).sum())
size = sum(p.stat().st_size for p in VEC.rglob('*.mem'))
print(f'{len(sel)} images, {len(list(TRC.glob("*.mem")))} trace files, {size/1e6:.1f} MB of vectors')
print(f'golden decision (T={DEFAULT_T}) agrees with ground truth on {agree}/{len(sel)} ({agree/len(sel):.1%}) '
      f'-- the RTL must match the LOGITS, including the {len(sel)-agree} mistakes')

## 16. Summary, manifest, download

`manifest.json` records everything needed to reproduce or audit this export: configuration,
split sizes, the patient IDs of the verification images, metrics, thresholds, widths, library
versions, the preprocessing self-test results, reference hashes for the board, and a SHA-256 of
every exported memory file.

### Where the files go in the repository
```
colab/ProjectFlash_V1.ipynb
v1/mem/<stage>/            weights.mem  bias.mem  layer_table.*  sim_config.vh  vectors/
tools/golden_model_v1.py   (from export/<stage>/tools/)
tools/flash_preprocess.py
docs/V1_<stage>_summary.md (paste the table below)
```
At 224×224 the vectors are tens of MB. Either commit them with Git LFS, or commit only the
weights/table files and regenerate vectors from `model_state.pt` (which `.gitignore` already excludes).

In [ ]:
for f in ['flash_preprocess.py', 'golden_model_v1.py']:
    shutil.copy(TOOLS / f, EXPORT / 'tools' / f)

sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
dflt = OPS[f'sens>={CFG["TARGET_SENS"]:.0%} on val (DEFAULT)']
summary = {
    'stage': STAGE + (' (SMOKE TEST - synthetic data, numbers meaningless)' if SMOKE_TEST else ''),
    'dataset': CFG['DATASET'], 'input': f'{IMG}x{IMG} uint8',
    'patients train/val/test': f'{len(tr_idx)}/{len(va_idx)}/{len(te_idx)}',
    'test prevalence': f'{Yte.mean():.1%}',
    'parameters': int(WFLAT.size + BFLAT.size), 'MACs/image': int(ARCH.MACs.sum()),
    'val AUROC': f'{VAL_AUC[0]:.4f} [{VAL_AUC[1]:.4f}, {VAL_AUC[2]:.4f}]',
    'TEST AUROC': f'{TEST_AUC[0]:.4f} [{TEST_AUC[1]:.4f}, {TEST_AUC[2]:.4f}]',
    'view-only AUROC (shortcut floor)': None if SHORTCUT_AUROC is None else round(SHORTCUT_AUROC, 4),
    'default T (chosen on val)': DEFAULT_T,
    'TEST sensitivity @T': f3(dflt['test']['sens']), 'TEST specificity @T': f3(dflt['test']['spec']),
    'TEST sens/spec @argmax (v0 style)': f'{OPS["argmax, T=0 (v0)"]["test"]["sens"][0]:.1%} / {OPS["argmax, T=0 (v0)"]["test"]["spec"][0]:.1%}',
    'golden == torch, intermediates': f'0 mismatches on {len(sel)} images' if CHECK1_OK else 'FAIL',
    'golden == torch, test logits': f'{CHECK2[0]}/{CHECK2[1]}',
    'conv engine accumulator bits': ENGINE_ACC_BITS, 'margin bits': MARGIN_BITS,
    'preprocessing self-test': results,
}
print('| | |\n|---|---|')
for k, v in summary.items(): print(f'| {k} | {v} |')

manifest = dict(summary=summary, cfg=CFG, preproc_version=fp.PREPROC_VERSION, preproc_reference_sha256=REF_HASHES,
                versions=dict(torch=torch.__version__, numpy=np.__version__, pydicom=pydicom.__version__, opencv=cv2.__version__),
                layers=layers, shifts={L['name']: L['shift'] for L in layers},
                thresholds={k: v['T'] for k, v in OPS.items()},
                acc_stats={k: list(v) for k, v in ACC_STATS.items()},
                subgroups=SUBGROUPS.to_dict(orient='records'),
                verification=dict(test_positions=sel.tolist(), patient_ids=mte['patientId'].values[sel].tolist(),
                                  ground_truth=Yte[sel].tolist(), class3=mte['class3'].values[sel].tolist()),
                file_sha256={str(p.relative_to(EXPORT)): sha(p) for p in sorted(EXPORT.glob('*.mem'))})
(EXPORT / 'manifest.json').write_text(json.dumps(manifest, indent=1, default=str))
(EXPORT / 'report' / 'summary.json').write_text(json.dumps(summary, indent=1, default=str))

zpath = shutil.make_archive(str(WORK / f'flash_{TAG}'), 'zip', EXPORT)
print(f'\n{sum(1 for _ in EXPORT.rglob("*") if _.is_file())} files -> {zpath} ({Path(zpath).stat().st_size/1e6:.1f} MB)')
if USE_DRIVE and IN_COLAB:
    shutil.copy(zpath, CACHE); print('copied to Drive:', CACHE)
if IN_COLAB:
    from google.colab import files; files.download(zpath)